# PhiShark2026 — Projet Big Data et Deep Learning

Ce notebook est une reconstruction complète depuis zéro selon CRISP-DM.
Aucun ancien script, rapport, fichier Parquet ou notebook n'est utilisé.

Étapes prévues :
1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling Big Data et Deep Learning
5. Evaluation
6. Deployment : dashboard et rapport PDF

## 0. Configuration de l'environnement

Cette cellule ne dépend d'aucun module créé précédemment. Elle définit uniquement les chemins du dataset brut.

In [ ]:
Partie A — CRISP-DM et données

In [1]:
# CELLULE 0 - Imports et configuration
from pathlib import Path
from urllib.parse import urlparse
import platform
import sys
import pandas as pd

DATASET_ROOT = (Path.cwd() / 'phisharkl2026dataset-v1.1').resolve()
METADATA_CSV = DATASET_ROOT / 'phishark2026_public_metadata.csv'

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset introuvable : {DATASET_ROOT}')
if not METADATA_CSV.is_file():
    raise FileNotFoundError(f'Metadonnees introuvables : {METADATA_CSV}')

print('Python :', sys.version)
print('Systeme :', platform.platform())
print('Dataset :', DATASET_ROOT)
print('Dossier present :', DATASET_ROOT.is_dir())


Python : 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 12:58:53) [MSC v.1929 64 bit (AMD64)]
Systeme : Windows-11-10.0.26200-SP0
Dataset : C:\Users\pc\Desktop\Extracted dataset\phisharkl2026dataset-v1.1
Dossier present : True


In [2]:
# CELLULE 1 - Inventaire du dataset brut
zip_files = list(DATASET_ROOT.glob('*.zip'))
schema_files = list((DATASET_ROOT / 'schemas').rglob('*'))

print('Nombre d archives ZIP :', len(zip_files))
print('Nombre de fichiers dans schemas :', sum(path.is_file() for path in schema_files))
if len(zip_files) != 67502:
    raise ValueError(f'Nombre d archives inattendu : {len(zip_files)}')
for path in sorted(schema_files):
    if path.is_file():
        print('-', path.relative_to(DATASET_ROOT))


Nombre d archives ZIP : 67502
Nombre de fichiers dans schemas : 11
- schemas\compliance_files.schema.json
- schemas\dns.schema.json
- schemas\geo_accessibility.schema.json
- schemas\ip_geo.schema.json
- schemas\ip_whois.schema.json
- schemas\metadata.schema.json
- schemas\ports.schema.json
- schemas\redirection.schema.json
- schemas\tls_certificate.schema.json
- schemas\web_page.schema.json
- schemas\whois.schema.json


In [3]:
# CELLULE 2 - Chargement des metadonnees publiques
metadata_df = pd.read_csv(METADATA_CSV)
print('Dimensions :', metadata_df.shape)
print('Labels :')
print(metadata_df['label'].value_counts(dropna=False))
print('Sources :')
print(metadata_df['label_source'].value_counts(dropna=False))
print('Statuts :')
print(metadata_df['scan_status'].value_counts(dropna=False))
print('Valeurs manquantes :')
display(metadata_df.isna().sum().sort_values(ascending=False).to_frame('missing_count'))


Dimensions : (67502, 24)
Labels :
label
benign      34115
phishing    33387
Name: count, dtype: int64
Sources :
label_source
benign_list    34115
openphish      30878
usom            2509
Name: count, dtype: int64
Statuts :
scan_status
completed    67386
timed_out      116
Name: count, dtype: int64
Valeurs manquantes :


,missing_count
free_hosting_provider,37057
page_status,11190
scan_id,0
archive_path,0
label_id,0
label_source,0
archive_filename,0
label,0
source_group,0
source_name,0


In [4]:
# CELLULE 3 - Data Understanding
print('Hotes uniques :', metadata_df['target_host'].nunique())
print('Domaines racines uniques :', metadata_df['root_domain'].nunique())
print('Domaines d enregistrement uniques :', metadata_df['registration_domain'].nunique())
availability = pd.DataFrame({
    'available': [metadata_df['has_html'].sum(), metadata_df['has_screenshot'].sum(), metadata_df['has_favicon'].sum()],
    'missing': [(~metadata_df['has_html']).sum(), (~metadata_df['has_screenshot']).sum(), (~metadata_df['has_favicon']).sum()]
}, index=['HTML', 'Screenshot', 'Favicon'])
availability['availability_percent'] = (availability['available'] / len(metadata_df) * 100).round(2)
display(availability)
domain_label_counts = metadata_df.groupby('registration_domain')['label'].nunique()
print('Domaines presents dans les deux classes :', int((domain_label_counts > 1).sum()))


Hotes uniques : 56662
Domaines racines uniques : 56062
Domaines d enregistrement uniques : 24908


,available,missing,availability_percent
HTML,66894,608,99.10
Screenshot,67502,0,100.00
Favicon,48158,19344,71.34


Domaines presents dans les deux classes : 101


In [5]:
# CELLULE 4 - Data Preparation et features URL

prepared_df = metadata_df.copy()

# Conversion des dates avec formats mixtes
for column in ["scan_created_at", "scan_finished_at"]:
    prepared_df[column] = pd.to_datetime(
        prepared_df[column],
        format="mixed",
        errors="coerce",
        utc=True
    )

# Conversion des variables booléennes
boolean_columns = [
    "is_free_hosting_provider",
    "has_screenshot",
    "has_html",
    "has_favicon"
]

for column in boolean_columns:
    prepared_df[column] = prepared_df[column].astype("boolean")

# Conversion du statut HTTP
prepared_df["http_status"] = pd.to_numeric(
    prepared_df["http_status"],
    errors="coerce"
)

# Durée conservée uniquement pour l'audit
prepared_df["scan_duration_seconds"] = (
    prepared_df["scan_finished_at"]
    - prepared_df["scan_created_at"]
).dt.total_seconds()

prepared_df["duration_is_valid"] = (
    prepared_df["scan_duration_seconds"] >= 0
)


def extract_url_features(row):
    target_url = str(row["target_url"])
    final_url = str(row["final_url"])

    parsed = urlparse(target_url)
    final_parsed = urlparse(final_url)

    hostname = parsed.hostname or ""
    path = parsed.path or ""
    query = parsed.query or ""

    suspicious_tokens = (
        "login",
        "verify",
        "verification",
        "account",
        "secure",
        "update",
        "password",
        "signin",
        "confirm",
        "wallet",
        "bank"
    )

    url_lower = target_url.lower()

    return {
        "url_length": len(target_url),

        "url_host_length": len(hostname),

        "url_path_length": len(path),

        "url_query_length": len(query),

        "url_path_depth": path.count("/"),

        "url_query_parameter_count": (
            query.count("&") + 1
            if query
            else 0
        ),

        "url_subdomain_count": max(
            hostname.count(".") - 1,
            0
        ),

        "url_has_https": int(
            parsed.scheme.lower() == "https"
        ),

        "url_has_query": int(
            bool(query)
        ),

        "url_has_fragment": int(
            bool(parsed.fragment)
        ),

        "url_has_at_symbol": int(
            "@" in target_url
        ),

        "url_has_ip_address": int(
            bool(
                parsed.hostname
                and parsed.hostname.replace(".", "").isdigit()
            )
        ),

        "url_digit_count": sum(
            character.isdigit()
            for character in target_url
        ),

        "url_special_character_count": sum(
            not character.isalnum()
            for character in target_url
        ),

        "url_suspicious_token_count": sum(
            token in url_lower
            for token in suspicious_tokens
        ),

        "url_redirect_changed_host": int(
            (parsed.hostname or "").lower()
            != (final_parsed.hostname or "").lower()
        ),

        "final_url_length": len(final_url),

        "http_status_numeric": row["http_status"],

        "has_html_numeric": int(
            bool(row["has_html"])
        ),

        "has_screenshot_numeric": int(
            bool(row["has_screenshot"])
        ),

        "has_favicon_numeric": int(
            bool(row["has_favicon"])
        ),

        "is_free_hosting_numeric": int(
            bool(row["is_free_hosting_provider"])
        )
    }


url_features = prepared_df.apply(
    extract_url_features,
    axis=1,
    result_type="expand"
)

print("Features URL :", url_features.shape)

print(
    "Dates créées manquantes :",
    prepared_df["scan_created_at"].isna().sum()
)

print(
    "Dates terminées manquantes :",
    prepared_df["scan_finished_at"].isna().sum()
)

print(
    "Durées valides :",
    int(prepared_df["duration_is_valid"].sum())
)

print(
    "Valeurs manquantes dans les features :",
    int(url_features.isna().sum().sum())
)

display(url_features.head())

Features URL : (67502, 22)
Dates créées manquantes : 0
Dates terminées manquantes : 0
Durées valides : 0
Valeurs manquantes dans les features : 0


,url_length,url_host_length,url_path_length,url_query_length,url_path_depth,url_query_parameter_count,url_subdomain_count,url_has_https,url_has_query,url_has_fragment,...,url_digit_count,url_special_character_count,url_suspicious_token_count,url_redirect_changed_host,final_url_length,http_status_numeric,has_html_numeric,has_screenshot_numeric,has_favicon_numeric,is_free_hosting_numeric
0,62.0,23.0,32.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,9.0,0.0,0.0,62.0,200.0,1.0,1.0,1.0,0.0
1,37.0,30.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,7.0,0.0,0.0,39.0,200.0,1.0,1.0,0.0,1.0
2,86.0,79.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,6.0,7.0,0.0,1.0,83.0,200.0,1.0,1.0,0.0,0.0
3,145.0,7.0,77.0,52.0,3.0,1.0,0.0,1.0,1.0,0.0,...,38.0,12.0,0.0,0.0,145.0,200.0,1.0,1.0,0.0,0.0
4,41.0,34.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,7.0,0.0,0.0,43.0,200.0,1.0,1.0,1.0,0.0


In [ ]:
Partie B — Extraction technique

In [6]:
# CELLULE 5 - Test d'extraction des features techniques
import json
from zipfile import ZipFile

EXPECTED_JSON_FILES = {
    "dns": "data/dns.json",
    "whois": "data/whois.json",
    "ip_geo": "data/ip_geo.json",
    "ports": "data/ports.json",
    "tls": "data/tls_certificate.json",
    "web_page": "data/web_page.json",
    "redirection": "data/redirection.json",
    "compliance": "data/compliance_files.json",
}


def safe_length(value):
    if isinstance(value, (str, list, dict)):
        return len(value)
    return 0


def read_optional_json(archive, member_name):
    if member_name not in archive.namelist():
        return {}, False

    with archive.open(member_name) as file:
        return json.load(file), True


def extract_technical_features(archive_path):
    with ZipFile(archive_path, "r") as archive:
        data = {}
        availability = {}

        for name, member_name in EXPECTED_JSON_FILES.items():
            value, available = read_optional_json(
                archive,
                member_name
            )

            data[name] = (
                value
                if isinstance(value, dict)
                else {}
            )

            availability[f"{name}_available"] = int(available)

    dns = data["dns"]
    whois = data["whois"]
    ip_geo = data["ip_geo"]
    ports = data["ports"]
    tls = data["tls"]
    web_page = data["web_page"]
    redirection = data["redirection"]
    compliance = data["compliance"]

    features = {
        "dns_has_records": int(bool(dns.get("records"))),
        "dns_record_count": safe_length(dns.get("records")),

        "whois_has_creation_date": int(
            bool(whois.get("creation_date"))
        ),
        "whois_has_expiration_date": int(
            bool(whois.get("expiration_date"))
        ),
        "whois_has_registrar": int(
            bool(whois.get("registrar"))
        ),
        "whois_name_server_count": safe_length(
            whois.get("name_servers")
        ),

        "ip_geo_has_ip": int(bool(ip_geo.get("ip"))),
        "ip_geo_has_country": int(
            bool(ip_geo.get("country_code"))
        ),

        "ports_count": safe_length(
            ports.get("open_ports")
        ),

        "tls_success": int(bool(tls.get("success"))),
        "tls_has_issuer": int(bool(tls.get("issuer"))),
        "tls_has_subject": int(bool(tls.get("subject"))),
        "tls_has_san": safe_length(
            tls.get("subject_alternative_names")
        ),

        "web_title_length": safe_length(
            web_page.get("title")
        ),
        "web_description_length": safe_length(
            web_page.get("description")
        ),
        "web_technology_count": safe_length(
            web_page.get("detected_technologies")
        ),
        "web_outgoing_link_count": safe_length(
            web_page.get("outgoing_links")
        ),
        "web_total_url_count": (
            web_page.get("total_url_count") or 0
        ),

        "redirect_count": (
            redirection.get("redirect_count") or 0
        ),
        "redirect_chain_length": safe_length(
            redirection.get("chain")
        ),

        "compliance_checked_count": safe_length(
            compliance.get("checked_files")
        ),
        "compliance_found_count": safe_length(
            compliance.get("found_files")
        ),
    }

    features.update(availability)
    return features


sample_archive = (
    DATASET_ROOT
    / metadata_df.iloc[0]["archive_filename"]
)

technical_sample = extract_technical_features(
    sample_archive
)

print(
    "Nombre de features techniques :",
    len(technical_sample)
)

display(
    pd.Series(
        technical_sample,
        name="value"
    ).to_frame()
)

Nombre de features techniques : 30


,value
dns_has_records,1
dns_record_count,3
whois_has_creation_date,1
whois_has_expiration_date,1
whois_has_registrar,1
whois_name_server_count,2
ip_geo_has_ip,1
ip_geo_has_country,1
ports_count,12
tls_success,1


In [7]:
# CELLULE 6 - Extraction technique complète

import time

all_technical_rows = []
technical_errors = []

all_archives = [
    DATASET_ROOT / filename
    for filename in metadata_df["archive_filename"]
]

start_time = time.perf_counter()

for index, archive_path in enumerate(all_archives, start=1):
    try:
        features = extract_technical_features(archive_path)
        features["archive_filename"] = archive_path.name
        all_technical_rows.append(features)

    except Exception as error:
        technical_errors.append({
            "archive_filename": archive_path.name,
            "error": repr(error)
        })

    if index % 500 == 0:
        elapsed = time.perf_counter() - start_time
        speed = index / elapsed
        remaining = (len(all_archives) - index) / speed

        print(
            f"{index:,}/{len(all_archives):,} archives | "
            f"{elapsed / 60:.1f} min écoulées | "
            f"environ {remaining / 60:.1f} min restantes"
        )

technical_df = pd.DataFrame(all_technical_rows)

print("\n=== EXTRACTION COMPLÈTE ===")
print("Archives traitées :", len(all_archives))
print("Lignes extraites :", len(technical_df))
print("Erreurs :", len(technical_errors))
print("Features techniques :", len(technical_df.columns))

if technical_errors:
    display(pd.DataFrame(technical_errors).head())

assert len(technical_df) + len(technical_errors) == len(all_archives)

500/67,502 archives | 0.0 min écoulées | environ 5.6 min restantes
1,000/67,502 archives | 0.1 min écoulées | environ 5.6 min restantes
1,500/67,502 archives | 0.1 min écoulées | environ 5.3 min restantes
2,000/67,502 archives | 0.2 min écoulées | environ 5.2 min restantes
2,500/67,502 archives | 0.2 min écoulées | environ 5.0 min restantes
3,000/67,502 archives | 0.2 min écoulées | environ 5.1 min restantes
3,500/67,502 archives | 0.3 min écoulées | environ 5.0 min restantes
4,000/67,502 archives | 0.3 min écoulées | environ 5.0 min restantes
4,500/67,502 archives | 0.4 min écoulées | environ 4.9 min restantes
5,000/67,502 archives | 0.4 min écoulées | environ 4.9 min restantes
5,500/67,502 archives | 0.4 min écoulées | environ 4.8 min restantes
6,000/67,502 archives | 0.5 min écoulées | environ 4.7 min restantes
6,500/67,502 archives | 0.5 min écoulées | environ 4.7 min restantes
7,000/67,502 archives | 0.5 min écoulées | environ 4.6 min restantes
7,500/67,502 archives | 0.6 min écou

In [8]:
# CELLULE 7 - Fusion des modalités

url_features_with_id = url_features.copy()
url_features_with_id["archive_filename"] = (
    metadata_df["archive_filename"].values
)

features_df = url_features_with_id.merge(
    technical_df,
    on="archive_filename",
    how="inner",
    validate="one_to_one"
)

features_df = features_df.merge(
    metadata_df[
        [
            "archive_filename",
            "scan_id",
            "label",
            "label_id",
            "registration_domain"
        ]
    ],
    on="archive_filename",
    how="inner",
    validate="one_to_one"
)

print("Dimensions finales :", features_df.shape)
print(
    "Archives uniques :",
    features_df["archive_filename"].nunique()
)
print(
    "Valeurs manquantes :",
    int(features_df.isna().sum().sum())
)

assert len(features_df) == 67502
assert features_df["archive_filename"].is_unique
assert features_df["scan_id"].is_unique

display(features_df.head())

Dimensions finales : (67502, 57)
Archives uniques : 67502
Valeurs manquantes : 0


,url_length,url_host_length,url_path_length,url_query_length,url_path_depth,url_query_parameter_count,url_subdomain_count,url_has_https,url_has_query,url_has_fragment,...,ip_geo_available,ports_available,tls_available,web_page_available,redirection_available,compliance_available,scan_id,label,label_id,registration_domain
0,62.0,23.0,32.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,1,1,1,1,1,1,ba3a375a-c550-4abf-9603-cf40235a90c0,phishing,1,oklahomaweddingpros.com
1,37.0,30.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1,1,1,1,1,1,2b226308-949e-403a-8ee9-23a5685fd7ab,phishing,1,vercel.app
2,86.0,79.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,1,1,1,1,1,1,e664b18a-41bd-418f-8535-80334275a873,phishing,1,infura-ipfs.io
3,145.0,7.0,77.0,52.0,3.0,1.0,0.0,1.0,1.0,0.0,...,1,1,1,1,1,1,93eb0e1d-8845-47a7-8a14-a3e8850273a7,phishing,1,ipfs.io
4,41.0,34.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0,0,1,1,1,1,103680ff-c924-4265-8888-70b874cabf83,phishing,1,partner-program-center.com


In [ ]:
Partie C — Split et baselines

In [9]:
# CELLULE 8 - Split groupé anti-fuite

from sklearn.model_selection import GroupShuffleSplit

features_df["split"] = "unknown"

groups = features_df["registration_domain"].fillna(
    "__missing_registration_domain__"
)

splitter_train = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temporary_idx = next(
    splitter_train.split(
        features_df,
        features_df["label_id"],
        groups=groups
    )
)

train_df = features_df.iloc[train_idx].copy()
temporary_df = features_df.iloc[temporary_idx].copy()
temporary_groups = groups.iloc[temporary_idx]

splitter_validation = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

validation_relative_idx, test_relative_idx = next(
    splitter_validation.split(
        temporary_df,
        temporary_df["label_id"],
        groups=temporary_groups
    )
)

validation_df = temporary_df.iloc[
    validation_relative_idx
].copy()

test_df = temporary_df.iloc[
    test_relative_idx
].copy()

train_df["split"] = "train"
validation_df["split"] = "validation"
test_df["split"] = "test"

features_df.loc[train_df.index, "split"] = "train"
features_df.loc[validation_df.index, "split"] = "validation"
features_df.loc[test_df.index, "split"] = "test"

print("Train :", len(train_df))
print("Validation :", len(validation_df))
print("Test :", len(test_df))

print("\nRépartition des labels :")
print(
    pd.DataFrame({
        "train": train_df["label"].value_counts(),
        "validation": validation_df["label"].value_counts(),
        "test": test_df["label"].value_counts()
    }).fillna(0).astype(int)
)

train_domains = set(train_df["registration_domain"].dropna())
validation_domains = set(
    validation_df["registration_domain"].dropna()
)
test_domains = set(test_df["registration_domain"].dropna())

print("\n=== CONTRÔLE ANTI-FUITE ===")
print("Train ∩ validation :", len(train_domains & validation_domains))
print("Train ∩ test :", len(train_domains & test_domains))
print("Validation ∩ test :", len(validation_domains & test_domains))

assert len(train_df) + len(validation_df) + len(test_df) == 67502
assert len(train_domains & validation_domains) == 0
assert len(train_domains & test_domains) == 0
assert len(validation_domains & test_domains) == 0

Train : 42210
Validation : 8194
Test : 17098

Répartition des labels :
          train  validation  test
label                            
benign    20503        3794  9818
phishing  21707        4400  7280

=== CONTRÔLE ANTI-FUITE ===
Train ∩ validation : 0
Train ∩ test : 0
Validation ∩ test : 0


In [10]:
# CELLULE 9 - Construction des matrices de features

url_feature_columns = [
    column
    for column in features_df.columns
    if column.startswith("url_")
    or column in [
        "final_url_length",
        "http_status_numeric",
        "has_html_numeric",
        "has_screenshot_numeric",
        "has_favicon_numeric",
        "is_free_hosting_numeric"
    ]
]

technical_feature_columns = [
    column
    for column in features_df.columns
    if column.startswith((
        "dns_",
        "whois_",
        "ip_geo_",
        "ports_",
        "tls_",
        "web_",
        "redirect_",
        "compliance_"
    ))
]

feature_columns = (
    url_feature_columns
    + technical_feature_columns
)

excluded_columns = {
    "archive_filename",
    "scan_id",
    "label",
    "label_id",
    "registration_domain",
    "split"
}

assert not excluded_columns.intersection(feature_columns)
assert len(feature_columns) == len(set(feature_columns))

X = features_df[feature_columns].copy()
y = features_df["label_id"].astype("int64")

# Conversion numérique finale
X = X.apply(pd.to_numeric, errors="coerce")

if X.isna().any().any():
    print("Valeurs manquantes détectées :")
    display(X.isna().sum().sort_values(ascending=False).head())
    raise ValueError("Des valeurs manquantes existent dans les features.")

# Masques des partitions
train_mask = features_df["split"].eq("train")
validation_mask = features_df["split"].eq("validation")
test_mask = features_df["split"].eq("test")

X_train = X.loc[train_mask]
X_validation = X.loc[validation_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_validation = y.loc[validation_mask]
y_test = y.loc[test_mask]

print("Features URL :", len(url_feature_columns))
print("Features techniques :", len(technical_feature_columns))
print("Features totales :", len(feature_columns))

print("\nTrain :", X_train.shape, y_train.shape)
print("Validation :", X_validation.shape, y_validation.shape)
print("Test :", X_test.shape, y_test.shape)

print("\nValeurs manquantes :", int(X.isna().sum().sum()))

display(pd.DataFrame({
    "modalite": ["URL / metadata", "Technical JSON", "Total"],
    "nombre_features": [
        len(url_feature_columns),
        len(technical_feature_columns),
        len(feature_columns)
    ]
}))

Features URL : 22
Features techniques : 29
Features totales : 51

Train : (42210, 51) (42210,)
Validation : (8194, 51) (8194,)
Test : (17098, 51) (17098,)

Valeurs manquantes : 0


,modalite,nombre_features
0,URL / metadata,22
1,Technical JSON,29
2,Total,51


In [11]:
# CELLULE 10 - Normalisation sans fuite

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

print("Train normalisé :", X_train_scaled.shape)
print("Validation normalisée :", X_validation_scaled.shape)
print("Test normalisé :", X_test_scaled.shape)

print("\nMoyenne approximative du train :",
      X_train_scaled.mean().round(4))

print("Écart-type approximatif du train :",
      X_train_scaled.std().round(4))

assert X_train_scaled.shape == (42210, 51)
assert X_validation_scaled.shape == (8194, 51)
assert X_test_scaled.shape == (17098, 51)

Train normalisé : (42210, 51)
Validation normalisée : (8194, 51)
Test normalisé : (17098, 51)

Moyenne approximative du train : -0.0
Écart-type approximatif du train : 0.9497


In [12]:
# CELLULE 11 - Baselines de classification

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
import time

models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )
}


def evaluate_model(model, X_eval, y_eval):
    probabilities = model.predict_proba(X_eval)[:, 1]
    predictions = (probabilities >= 0.5).astype("int64")

    tn, fp, fn, tp = confusion_matrix(
        y_eval,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "accuracy": accuracy_score(y_eval, predictions),
        "precision_phishing": precision_score(
            y_eval,
            predictions,
            zero_division=0
        ),
        "recall_phishing": recall_score(
            y_eval,
            predictions,
            zero_division=0
        ),
        "f1_phishing": f1_score(
            y_eval,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_eval,
            probabilities
        ),
        "pr_auc": average_precision_score(
            y_eval,
            probabilities
        ),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    }


validation_results = []
fitted_models = {}

for model_name, model in models.items():
    start_time = time.perf_counter()

    model.fit(
        X_train_scaled,
        y_train
    )

    training_seconds = time.perf_counter() - start_time

    metrics = evaluate_model(
        model,
        X_validation_scaled,
        y_validation
    )

    metrics["model"] = model_name
    metrics["training_seconds"] = round(
        training_seconds,
        3
    )

    validation_results.append(metrics)
    fitted_models[model_name] = model

validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

display(validation_results_df)

,accuracy,precision_phishing,recall_phishing,f1_phishing,roc_auc,pr_auc,true_negatives,false_positives,false_negatives,true_positives,model,training_seconds
0,0.935074,0.953565,0.924091,0.938596,0.981373,0.986394,3596,198,334,4066,hist_gradient_boosting,6.208
1,0.843300,0.922222,0.773409,0.841286,0.934687,0.948716,3507,287,997,3403,logistic_regression,0.663


In [13]:
# CELLULE 12 - Optimisation du seuil sur validation

import numpy as np
from sklearn.metrics import precision_recall_curve

selected_model = fitted_models["hist_gradient_boosting"]

validation_probabilities = selected_model.predict_proba(
    X_validation_scaled
)[:, 1]

threshold_rows = []

for threshold in np.arange(0.10, 0.96, 0.05):
    validation_predictions = (
        validation_probabilities >= threshold
    ).astype("int64")

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        validation_predictions,
        labels=[0, 1]
    ).ravel()

    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "precision_phishing": precision_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        "recall_phishing": recall_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        "f1_phishing": f1_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "true_negatives": tn
    })

threshold_results_df = pd.DataFrame(threshold_rows)

display(
    threshold_results_df.sort_values(
        "f1_phishing",
        ascending=False
    ).head(10)
)

,threshold,precision_phishing,recall_phishing,f1_phishing,false_positives,false_negatives,true_positives,true_negatives
9,0.55,0.962521,0.916364,0.938875,157,368,4032,3637
8,0.50,0.953565,0.924091,0.938596,198,334,4066,3596
10,0.60,0.971304,0.907727,0.938440,118,406,3994,3676
7,0.45,0.942910,0.930909,0.936871,248,304,4096,3546
11,0.65,0.976555,0.899318,0.936346,95,443,3957,3699
6,0.40,0.925760,0.940909,0.933273,332,260,4140,3462
12,0.70,0.980353,0.884545,0.929988,78,508,3892,3716
5,0.35,0.910425,0.944773,0.927281,409,243,4157,3385
13,0.75,0.986036,0.866591,0.922463,54,587,3813,3740
14,0.80,0.990005,0.855455,0.917825,38,636,3764,3756


In [14]:
# CELLULE 13 - Sélection du seuil et évaluation finale groupée

best_threshold_row = (
    threshold_results_df
    .sort_values(
        ["f1_phishing", "recall_phishing"],
        ascending=[False, False]
    )
    .iloc[0]
)

selected_threshold = float(
    best_threshold_row["threshold"]
)

test_probabilities = selected_model.predict_proba(
    X_test_scaled
)[:, 1]

test_predictions = (
    test_probabilities >= selected_threshold
).astype("int64")

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1]
).ravel()

grouped_test_results = pd.DataFrame([{
    "protocol": "grouped_registration_domain",
    "model": "hist_gradient_boosting",
    "threshold_selected_on_validation": selected_threshold,
    "accuracy": accuracy_score(y_test, test_predictions),
    "precision_phishing": precision_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "recall_phishing": recall_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "f1_phishing": f1_score(
        y_test,
        test_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        test_probabilities
    ),
    "pr_auc": average_precision_score(
        y_test,
        test_probabilities
    ),
    "true_negatives": tn,
    "false_positives": fp,
    "false_negatives": fn,
    "true_positives": tp
}])

display(grouped_test_results)

,protocol,model,threshold_selected_on_validation,accuracy,precision_phishing,recall_phishing,f1_phishing,roc_auc,pr_auc,true_negatives,false_positives,false_negatives,true_positives
0,grouped_registration_domain,hist_gradient_boosting,0.55,0.97257,0.976227,0.958929,0.967501,0.987738,0.989894,9648,170,299,6981


In [15]:
# CELLULE 14 - Comparaison unimodale URL et technique

from sklearn.base import clone

modality_results = []

modality_sets = {
    "url_metadata_only": url_feature_columns,
    "technical_json_only": technical_feature_columns,
    "url_plus_technical": feature_columns,
}

for modality_name, columns in modality_sets.items():
    X_modality = features_df[columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    X_modality_train = X_modality.loc[train_mask]
    X_modality_validation = X_modality.loc[validation_mask]
    X_modality_test = X_modality.loc[test_mask]

    modality_scaler = StandardScaler()

    X_modality_train_scaled = modality_scaler.fit_transform(
        X_modality_train
    )
    X_modality_validation_scaled = modality_scaler.transform(
        X_modality_validation
    )
    X_modality_test_scaled = modality_scaler.transform(
        X_modality_test
    )

    modality_model = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    start_time = time.perf_counter()

    modality_model.fit(
        X_modality_train_scaled,
        y_train
    )

    training_seconds = time.perf_counter() - start_time

    validation_probabilities = modality_model.predict_proba(
        X_modality_validation_scaled
    )[:, 1]

    validation_predictions = (
        validation_probabilities >= 0.5
    ).astype("int64")

    modality_results.append({
        "modality": modality_name,
        "feature_count": len(columns),
        "accuracy": accuracy_score(
            y_validation,
            validation_predictions
        ),
        "precision_phishing": precision_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        "recall_phishing": recall_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        "f1_phishing": f1_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_validation,
            validation_probabilities
        ),
        "pr_auc": average_precision_score(
            y_validation,
            validation_probabilities
        ),
        "training_seconds": round(training_seconds, 3)
    })

modality_results_df = (
    pd.DataFrame(modality_results)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

display(modality_results_df)

,modality,feature_count,accuracy,precision_phishing,recall_phishing,f1_phishing,roc_auc,pr_auc,training_seconds
0,url_plus_technical,51,0.935074,0.953565,0.924091,0.938596,0.981373,0.986394,2.260
1,technical_json_only,29,0.884794,0.909867,0.871818,0.890436,0.949967,0.961831,1.481
2,url_metadata_only,22,0.845985,0.863026,0.847727,0.855308,0.928261,0.947536,1.424


In [16]:
# CELLULE 15 - Diagnostic overfitting / underfitting

selected_model = fitted_models["hist_gradient_boosting"]

def evaluate_probabilities(y_true, probabilities, threshold=0.55):
    predictions = (
        probabilities >= threshold
    ).astype("int64")

    return {
        "accuracy": accuracy_score(
            y_true, predictions
        ),
        "precision_phishing": precision_score(
            y_true, predictions,
            zero_division=0
        ),
        "recall_phishing": recall_score(
            y_true, predictions,
            zero_division=0
        ),
        "f1_phishing": f1_score(
            y_true, predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_true, probabilities
        ),
        "pr_auc": average_precision_score(
            y_true, probabilities
        )
    }

train_probabilities = selected_model.predict_proba(
    X_train_scaled
)[:, 1]

validation_probabilities = selected_model.predict_proba(
    X_validation_scaled
)[:, 1]

test_probabilities = selected_model.predict_proba(
    X_test_scaled
)[:, 1]

fit_diagnostic = pd.DataFrame([
    {
        "partition": "train",
        **evaluate_probabilities(
            y_train,
            train_probabilities,
            selected_threshold
        )
    },
    {
        "partition": "validation",
        **evaluate_probabilities(
            y_validation,
            validation_probabilities,
            selected_threshold
        )
    },
    {
        "partition": "test_grouped",
        **evaluate_probabilities(
            y_test,
            test_probabilities,
            selected_threshold
        )
    }
])

display(fit_diagnostic)

train_pr_auc = fit_diagnostic.loc[
    fit_diagnostic["partition"] == "train",
    "pr_auc"
].iloc[0]

validation_pr_auc = fit_diagnostic.loc[
    fit_diagnostic["partition"] == "validation",
    "pr_auc"
].iloc[0]

print(
    "Écart PR-AUC train-validation :",
    round(train_pr_auc - validation_pr_auc, 4)
)

,partition,accuracy,precision_phishing,recall_phishing,f1_phishing,roc_auc,pr_auc
0,train,0.966619,0.983240,0.951306,0.967009,0.994860,0.995681
1,validation,0.935929,0.962521,0.916364,0.938875,0.981373,0.986394
2,test_grouped,0.972570,0.976227,0.958929,0.967501,0.987738,0.989894


Écart PR-AUC train-validation : 0.0093


In [ ]:
Partie D — Robustesse temporelle

In [23]:
# CELLULE 16 - Split temporel corrigé

from sklearn.model_selection import StratifiedShuffleSplit

temporal_dates = prepared_df[
    ["scan_id", "scan_created_at"]
].copy()

temporal_df = features_df.merge(
    temporal_dates,
    on="scan_id",
    how="left",
    validate="one_to_one"
)

missing_dates = temporal_df["scan_created_at"].isna().sum()
print("Dates manquantes après fusion :", missing_dates)

if missing_dates > 0:
    raise ValueError(f"{missing_dates} dates manquantes.")

temporal_df = temporal_df.sort_values(
    "scan_created_at"
).reset_index(drop=True)

historical_end = int(len(temporal_df) * 0.80)

historical_df = temporal_df.iloc[:historical_end].copy()
temporal_test_df = temporal_df.iloc[historical_end:].copy()

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=2028
)

train_relative, validation_relative = next(
    splitter.split(
        historical_df,
        historical_df["label_id"]
    )
)

temporal_train_df = historical_df.iloc[train_relative].copy()
temporal_validation_df = historical_df.iloc[validation_relative].copy()

print("=== SPLIT TEMPOREL CORRIGÉ ===")
print("Train historique :", len(temporal_train_df))
print("Validation historique :", len(temporal_validation_df))
print("Test futur strict :", len(temporal_test_df))

print("\n=== RÉPARTITION DES LABELS ===")

display(
    pd.DataFrame({
        "train": temporal_train_df["label"].value_counts(),
        "validation": temporal_validation_df["label"].value_counts(),
        "test": temporal_test_df["label"].value_counts()
    }).fillna(0).astype(int)
)

Dates manquantes après fusion : 0
=== SPLIT TEMPOREL CORRIGÉ ===
Train historique : 43200
Validation historique : 10801
Test futur strict : 13501

=== RÉPARTITION DES LABELS ===


,train,validation,test
label,,,
benign,22394,5599,6122
phishing,20806,5202,7379


In [24]:
# CELLULE 17 - Préparation des matrices temporelles

# Création explicite du mapping scan_id -> partition temporelle
temporal_split_map = pd.concat([
    temporal_train_df[["scan_id"]].assign(
        temporal_split="train_temporal"
    ),
    temporal_validation_df[["scan_id"]].assign(
        temporal_split="validation_temporal"
    ),
    temporal_test_df[["scan_id"]].assign(
        temporal_split="test_temporal"
    )
], ignore_index=True)

assert temporal_split_map["scan_id"].is_unique
assert len(temporal_split_map) == len(features_df)

features_temporal_df = features_df.merge(
    temporal_split_map,
    on="scan_id",
    how="inner",
    validate="one_to_one"
)

assert len(features_temporal_df) == len(features_df)
assert features_temporal_df["temporal_split"].notna().all()

print("Répartition temporelle :")
print(features_temporal_df["temporal_split"].value_counts())

# Matrices de features
X_temporal = features_temporal_df[
    feature_columns
].apply(pd.to_numeric, errors="coerce")

y_temporal = features_temporal_df["label_id"].astype("int64")

if X_temporal.isna().any().any():
    raise ValueError("Des valeurs manquantes existent dans X_temporal.")

# Masques temporels
temporal_train_mask = (
    features_temporal_df["temporal_split"]
    == "train_temporal"
)

temporal_validation_mask = (
    features_temporal_df["temporal_split"]
    == "validation_temporal"
)

temporal_test_mask = (
    features_temporal_df["temporal_split"]
    == "test_temporal"
)

X_temporal_train = X_temporal.loc[temporal_train_mask]
X_temporal_validation = X_temporal.loc[
    temporal_validation_mask
]
X_temporal_test = X_temporal.loc[
    temporal_test_mask
]

y_temporal_train = y_temporal.loc[temporal_train_mask]
y_temporal_validation = y_temporal.loc[
    temporal_validation_mask
]
y_temporal_test = y_temporal.loc[
    temporal_test_mask
]

print("\nDimensions temporelles :")
print("Train :", X_temporal_train.shape, y_temporal_train.shape)
print(
    "Validation :",
    X_temporal_validation.shape,
    y_temporal_validation.shape
)
print("Test :", X_temporal_test.shape, y_temporal_test.shape)

Répartition temporelle :
temporal_split
train_temporal         43200
test_temporal          13501
validation_temporal    10801
Name: count, dtype: int64

Dimensions temporelles :
Train : (43200, 51) (43200,)
Validation : (10801, 51) (10801,)
Test : (13501, 51) (13501,)


In [25]:
# CELLULE 18 - Entraînement du modèle temporel

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

temporal_scaler = StandardScaler()

X_temporal_train_scaled = temporal_scaler.fit_transform(
    X_temporal_train
)

X_temporal_validation_scaled = temporal_scaler.transform(
    X_temporal_validation
)

X_temporal_test_scaled = temporal_scaler.transform(
    X_temporal_test
)

temporal_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

temporal_model.fit(
    X_temporal_train_scaled,
    y_temporal_train
)

temporal_validation_probabilities = (
    temporal_model
    .predict_proba(X_temporal_validation_scaled)[:, 1]
)

temporal_test_probabilities = (
    temporal_model
    .predict_proba(X_temporal_test_scaled)[:, 1]
)

print("Modèle temporel entraîné.")
print("Train :", X_temporal_train_scaled.shape)
print("Validation :", X_temporal_validation_scaled.shape)
print("Test :", X_temporal_test_scaled.shape)

Modèle temporel entraîné.
Train : (43200, 51)
Validation : (10801, 51)
Test : (13501, 51)


In [26]:
# CELLULE 19 - Évaluation temporelle finale

temporal_threshold_rows = []

for threshold in np.arange(0.10, 0.96, 0.05):
    predictions = (
        temporal_validation_probabilities >= threshold
    ).astype("int64")

    temporal_threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "precision_phishing": precision_score(
            y_temporal_validation,
            predictions,
            zero_division=0
        ),
        "recall_phishing": recall_score(
            y_temporal_validation,
            predictions,
            zero_division=0
        ),
        "f1_phishing": f1_score(
            y_temporal_validation,
            predictions,
            zero_division=0
        )
    })

temporal_threshold_df = pd.DataFrame(
    temporal_threshold_rows
)

best_temporal_threshold = float(
    temporal_threshold_df
    .sort_values(
        ["f1_phishing", "recall_phishing"],
        ascending=[False, False]
    )
    .iloc[0]["threshold"]
)

temporal_test_predictions = (
    temporal_test_probabilities >= best_temporal_threshold
).astype("int64")

tn, fp, fn, tp = confusion_matrix(
    y_temporal_test,
    temporal_test_predictions,
    labels=[0, 1]
).ravel()

temporal_test_results = pd.DataFrame([{
    "protocol": "temporal",
    "threshold": best_temporal_threshold,
    "accuracy": accuracy_score(
        y_temporal_test,
        temporal_test_predictions
    ),
    "precision_phishing": precision_score(
        y_temporal_test,
        temporal_test_predictions,
        zero_division=0
    ),
    "recall_phishing": recall_score(
        y_temporal_test,
        temporal_test_predictions,
        zero_division=0
    ),
    "f1_phishing": f1_score(
        y_temporal_test,
        temporal_test_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_temporal_test,
        temporal_test_probabilities
    ),
    "pr_auc": average_precision_score(
        y_temporal_test,
        temporal_test_probabilities
    ),
    "true_negatives": tn,
    "false_positives": fp,
    "false_negatives": fn,
    "true_positives": tp
}])

print("Seuil temporel sélectionné :", best_temporal_threshold)

display(
    temporal_threshold_df
    .sort_values("f1_phishing", ascending=False)
    .head(10)
)

display(temporal_test_results)

Seuil temporel sélectionné : 0.55


,threshold,precision_phishing,recall_phishing,f1_phishing
9,0.55,0.978602,0.958285,0.968337
7,0.45,0.969726,0.966744,0.968233
8,0.50,0.974289,0.961553,0.967879
10,0.60,0.981207,0.953479,0.967144
11,0.65,0.984824,0.948097,0.966112
6,0.40,0.963112,0.968666,0.965881
12,0.70,0.989699,0.941945,0.965232
5,0.35,0.955522,0.974625,0.964979
4,0.30,0.948517,0.977509,0.962795
13,0.75,0.991033,0.934833,0.962113


,protocol,threshold,accuracy,precision_phishing,recall_phishing,f1_phishing,roc_auc,pr_auc,true_negatives,false_positives,false_negatives,true_positives
0,temporal,0.55,0.665951,0.638105,0.898225,0.746144,0.815356,0.877169,2363,3759,751,6628


In [27]:
# CELLULE 20 - Diagnostic du distribution shift temporel

print("=== DISTRIBUTION DES LABELS ===")

temporal_label_table = pd.DataFrame({
    "train_temporal": y_temporal_train.value_counts(),
    "validation_temporal": y_temporal_validation.value_counts(),
    "test_temporal": y_temporal_test.value_counts()
}).fillna(0).astype(int)

display(temporal_label_table)

print("\nPourcentage phishing :")

for name, labels in [
    ("train_temporal", y_temporal_train),
    ("validation_temporal", y_temporal_validation),
    ("test_temporal", y_temporal_test)
]:
    print(
        name,
        ":",
        round(labels.mean() * 100, 2),
        "%"
    )

print("\n=== DISTRIBUTION DES PROBABILITÉS ===")

validation_probability_summary = (
    pd.Series(
        temporal_validation_probabilities,
        name="validation_probability"
    )
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

test_probability_summary = (
    pd.Series(
        temporal_test_probabilities,
        name="test_probability"
    )
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

probability_summary = pd.concat(
    [
        validation_probability_summary,
        test_probability_summary
    ],
    axis=1
)

display(probability_summary)

=== DISTRIBUTION DES LABELS ===


,train_temporal,validation_temporal,test_temporal
label_id,,,
0,22394,5599,6122
1,20806,5202,7379



Pourcentage phishing :
train_temporal : 48.16 %
validation_temporal : 48.16 %
test_temporal : 54.66 %

=== DISTRIBUTION DES PROBABILITÉS ===


,validation_probability,test_probability
count,10801.000000,13501.000000
mean,0.481210,0.762580
std,0.468735,0.318749
min,0.000080,0.001012
1%,0.000685,0.018546
5%,0.001009,0.061152
25%,0.007170,0.599957
50%,0.264517,0.939430
75%,0.997485,0.995151
95%,0.999902,0.999901


In [32]:
%pip install torch --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/126.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/126.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/126.6 MB ? eta -:--:--
   ---------------------------------------- 0.8/126.6 MB 1.3 MB/s eta 0:01:34
   ---------------------------------------- 0.8/126.6 MB 1.3 MB/s eta 0:01:34
   ---------------------------------------- 1.0/126.6 MB 1.2 MB/s eta 0:01:48
   ---------------------------------------- 1.3/126.6 MB 1.2 MB/s eta 0:01:41
   ---------------------------------------- 1.3/126.6 MB 1.2 MB/s eta 0:01:41
   ---------------------------------------- 1.3/126.6 MB 1.2 MB/s eta 0:01:41
   ------

In [ ]:
Partie E — Deep Learning MLP actuel

In [28]:
# CELLULE 22 - Deep Learning : MLP de fusion URL + signaux techniques

import copy
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)
X_validation_tensor = torch.tensor(
    X_validation_scaled,
    dtype=torch.float32
)
X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32
).view(-1, 1)

y_validation_tensor = torch.tensor(
    y_validation.to_numpy(),
    dtype=torch.float32
).view(-1, 1)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32
).view(-1, 1)

train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=512,
    shuffle=True
)


class MultimodalMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.25),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.20),

            nn.Linear(64, 1)
        )

    def forward(self, inputs):
        return self.network(inputs)


model = MultimodalMLP(
    input_dim=X_train_tensor.shape[1]
).to(device)

positive_count = float(y_train.sum())
negative_count = float(len(y_train) - positive_count)

pos_weight = torch.tensor(
    [negative_count / positive_count],
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

best_state = None
best_validation_pr_auc = -1.0
patience = 8
epochs_without_improvement = 0
history = []

for epoch in range(1, 51):
    model.train()
    epoch_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(batch_x)

    model.eval()

    with torch.no_grad():
        validation_logits = model(
            X_validation_tensor.to(device)
        )

        validation_probabilities_mlp = (
            torch.sigmoid(validation_logits)
            .cpu()
            .numpy()
            .ravel()
        )

    validation_pr_auc_mlp = average_precision_score(
        y_validation,
        validation_probabilities_mlp
    )

    history.append({
        "epoch": epoch,
        "loss": epoch_loss / len(X_train_tensor),
        "validation_pr_auc": validation_pr_auc_mlp
    })

    print(
        f"Epoch {epoch:02d} | "
        f"loss={epoch_loss / len(X_train_tensor):.4f} | "
        f"validation PR-AUC={validation_pr_auc_mlp:.4f}"
    )

    if validation_pr_auc_mlp > best_validation_pr_auc:
        best_validation_pr_auc = validation_pr_auc_mlp
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break

model.load_state_dict(best_state)

model.eval()

with torch.no_grad():
    test_probabilities_mlp = (
        torch.sigmoid(
            model(X_test_tensor.to(device))
        )
        .cpu()
        .numpy()
        .ravel()
    )

mlp_test_pr_auc = average_precision_score(
    y_test,
    test_probabilities_mlp
)

mlp_test_roc_auc = roc_auc_score(
    y_test,
    test_probabilities_mlp
)

print("\n=== MLP FINAL ===")
print("Validation PR-AUC :", round(best_validation_pr_auc, 6))
print("Test PR-AUC :", round(mlp_test_pr_auc, 6))
print("Test ROC-AUC :", round(mlp_test_roc_auc, 6))

Device utilisé : cpu
Epoch 01 | loss=0.3604 | validation PR-AUC=0.9510
Epoch 02 | loss=0.2568 | validation PR-AUC=0.9552
Epoch 03 | loss=0.2365 | validation PR-AUC=0.9624
Epoch 04 | loss=0.2234 | validation PR-AUC=0.9666
Epoch 05 | loss=0.2148 | validation PR-AUC=0.9700
Epoch 06 | loss=0.2083 | validation PR-AUC=0.9708
Epoch 07 | loss=0.2026 | validation PR-AUC=0.9723
Epoch 08 | loss=0.1973 | validation PR-AUC=0.9724
Epoch 09 | loss=0.1939 | validation PR-AUC=0.9746
Epoch 10 | loss=0.1886 | validation PR-AUC=0.9741
Epoch 11 | loss=0.1860 | validation PR-AUC=0.9753
Epoch 12 | loss=0.1819 | validation PR-AUC=0.9767
Epoch 13 | loss=0.1789 | validation PR-AUC=0.9765
Epoch 14 | loss=0.1773 | validation PR-AUC=0.9768
Epoch 15 | loss=0.1764 | validation PR-AUC=0.9766
Epoch 16 | loss=0.1720 | validation PR-AUC=0.9757
Epoch 17 | loss=0.1718 | validation PR-AUC=0.9776
Epoch 18 | loss=0.1700 | validation PR-AUC=0.9782
Epoch 19 | loss=0.1680 | validation PR-AUC=0.9752
Epoch 20 | loss=0.1664 | vali

In [29]:
# CELLULE 23 - Évaluation complète du MLP

threshold_rows = []

for threshold in np.arange(0.10, 0.96, 0.05):
    validation_predictions = (
        validation_probabilities_mlp >= threshold
    ).astype("int64")

    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "f1_phishing": f1_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        "recall_phishing": recall_score(
            y_validation,
            validation_predictions,
            zero_division=0
        ),
        "precision_phishing": precision_score(
            y_validation,
            validation_predictions,
            zero_division=0
        )
    })

mlp_threshold_df = pd.DataFrame(threshold_rows)

mlp_best_threshold = float(
    mlp_threshold_df
    .sort_values(
        ["f1_phishing", "recall_phishing"],
        ascending=[False, False]
    )
    .iloc[0]["threshold"]
)

mlp_test_predictions = (
    test_probabilities_mlp >= mlp_best_threshold
).astype("int64")

tn, fp, fn, tp = confusion_matrix(
    y_test,
    mlp_test_predictions,
    labels=[0, 1]
).ravel()

mlp_test_results = pd.DataFrame([{
    "model": "multimodal_mlp",
    "threshold": mlp_best_threshold,
    "accuracy": accuracy_score(y_test, mlp_test_predictions),
    "precision_phishing": precision_score(
        y_test,
        mlp_test_predictions,
        zero_division=0
    ),
    "recall_phishing": recall_score(
        y_test,
        mlp_test_predictions,
        zero_division=0
    ),
    "f1_phishing": f1_score(
        y_test,
        mlp_test_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        test_probabilities_mlp
    ),
    "pr_auc": average_precision_score(
        y_test,
        test_probabilities_mlp
    ),
    "true_negatives": tn,
    "false_positives": fp,
    "false_negatives": fn,
    "true_positives": tp
}])

print("Seuil MLP sélectionné :", mlp_best_threshold)
display(mlp_test_results)

Seuil MLP sélectionné : 0.5


,model,threshold,accuracy,precision_phishing,recall_phishing,f1_phishing,roc_auc,pr_auc,true_negatives,false_positives,false_negatives,true_positives
0,multimodal_mlp,0.5,0.961399,0.962808,0.945879,0.954268,0.982755,0.985493,9552,266,394,6886


In [ ]:
Partie F — Extraction HTML et screenshots

In [30]:
# CELLULE 25 - Test HTML + Screenshot

import gzip
import io
import re
import html as html_module
from zipfile import ZipFile
from html.parser import HTMLParser
from PIL import Image


class HTMLFeatureParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.title_text = []
        self.text_chunks = []
        self.forms = 0
        self.password_inputs = 0
        self.scripts = 0
        self.iframes = 0
        self.images = 0
        self.links = 0
        self.external_links = 0
        self.meta_tags = 0
        self.hidden_inputs = 0
        self.inline_script_length = 0
        self.current_tag = None

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        attributes = dict(attrs)

        if tag == "title":
            self.current_tag = "title"

        if tag == "form":
            self.forms += 1

        if tag == "input":
            input_type = str(attributes.get("type", "")).lower()

            if input_type == "password":
                self.password_inputs += 1

            if input_type == "hidden":
                self.hidden_inputs += 1

        if tag == "script":
            self.scripts += 1
            script_src = attributes.get("src")

            if not script_src:
                self.current_tag = "script"

        if tag == "iframe":
            self.iframes += 1

        if tag == "img":
            self.images += 1

        if tag == "a":
            self.links += 1

        if tag == "meta":
            self.meta_tags += 1

    def handle_endtag(self, tag):
        if tag.lower() in ["title", "script"]:
            self.current_tag = None

    def handle_data(self, data):
        if not data.strip():
            return

        self.text_chunks.append(data)

        if self.current_tag == "title":
            self.title_text.append(data)

        if self.current_tag == "script":
            self.inline_script_length += len(data)


def safe_read_json_from_archive(archive, member_name):
    if member_name not in archive.namelist():
        return None

    with archive.open(member_name) as file:
        return json.load(file)


def extract_html_features_from_bytes(html_bytes):
    try:
        html_text = gzip.decompress(html_bytes).decode(
            "utf-8",
            errors="replace"
        )
    except OSError:
        html_text = html_bytes.decode(
            "utf-8",
            errors="replace"
        )

    parser = HTMLFeatureParser()

    try:
        parser.feed(html_text)
    except Exception:
        pass

    html_lower = html_text.lower()

    suspicious_html_tokens = [
        "login",
        "password",
        "verify",
        "verification",
        "account",
        "secure",
        "confirm",
        "wallet",
        "bank",
        "captcha"
    ]

    return {
        "html_available": 1,
        "html_bytes": len(html_bytes),
        "html_text_length": len(html_text),
        "html_title_length": len("".join(parser.title_text)),
        "html_text_length_visible": len(" ".join(parser.text_chunks)),
        "html_form_count": parser.forms,
        "html_password_input_count": parser.password_inputs,
        "html_hidden_input_count": parser.hidden_inputs,
        "html_script_count": parser.scripts,
        "html_iframe_count": parser.iframes,
        "html_image_count": parser.images,
        "html_link_count": parser.links,
        "html_meta_count": parser.meta_tags,
        "html_inline_script_length": parser.inline_script_length,
        "html_suspicious_token_count": sum(
            token in html_lower
            for token in suspicious_html_tokens
        ),
        "html_has_login_form": int(
            parser.forms > 0
            and parser.password_inputs > 0
        ),
        "html_has_external_form_action": int(
            bool(
                re.search(
                    r'<form[^>]+action\s*=\s*["\']https?://',
                    html_lower
                )
            )
        )
    }


def extract_image_features_from_bytes(image_bytes):
    try:
        with Image.open(io.BytesIO(image_bytes)) as image:
            width, height = image.size
            channels = len(image.getbands())
            image_format = image.format or ""

        return {
            "screenshot_available": 1,
            "screenshot_bytes": len(image_bytes),
            "screenshot_width": width,
            "screenshot_height": height,
            "screenshot_channels": channels,
            "screenshot_aspect_ratio": (
                width / height
                if height > 0
                else 0.0
            ),
            "screenshot_format_png": int(
                image_format.upper() == "PNG"
            )
        }

    except Exception as error:
        return {
            "screenshot_available": 0,
            "screenshot_bytes": len(image_bytes),
            "screenshot_width": 0,
            "screenshot_height": 0,
            "screenshot_channels": 0,
            "screenshot_aspect_ratio": 0.0,
            "screenshot_format_png": 0,
            "screenshot_error": repr(error)
        }


def extract_html_image_features(archive_path):
    with ZipFile(archive_path, "r") as archive:
        members = set(archive.namelist())

        if "artifacts/page.html.gz" in members:
            with archive.open("artifacts/page.html.gz") as file:
                html_features = extract_html_features_from_bytes(
                    file.read()
                )
        else:
            html_features = {
                "html_available": 0
            }

        if "artifacts/screenshot.png" in members:
            with archive.open("artifacts/screenshot.png") as file:
                image_features = extract_image_features_from_bytes(
                    file.read()
                )
        else:
            image_features = {
                "screenshot_available": 0
            }

    return {
        **html_features,
        **image_features
    }


sample_html_image_rows = []
sample_html_image_errors = []

sample_archives = [
    DATASET_ROOT / filename
    for filename in metadata_df.head(100)["archive_filename"]
]

for index, archive_path in enumerate(sample_archives, start=1):
    try:
        row = extract_html_image_features(archive_path)
        row["archive_filename"] = archive_path.name
        sample_html_image_rows.append(row)
    except Exception as error:
        sample_html_image_errors.append({
            "archive_filename": archive_path.name,
            "error": repr(error)
        })

    if index % 10 == 0:
        print(f"{index}/100 archives analysées")

html_image_sample_df = pd.DataFrame(
    sample_html_image_rows
)

print("\n=== TEST HTML + SCREENSHOT ===")
print("Lignes extraites :", len(html_image_sample_df))
print("Erreurs :", len(sample_html_image_errors))
print("Features :", len(html_image_sample_df.columns))

if sample_html_image_errors:
    display(pd.DataFrame(sample_html_image_errors).head())

display(html_image_sample_df.head())

10/100 archives analysées
20/100 archives analysées
30/100 archives analysées
40/100 archives analysées
50/100 archives analysées
60/100 archives analysées
70/100 archives analysées
80/100 archives analysées
90/100 archives analysées
100/100 archives analysées

=== TEST HTML + SCREENSHOT ===
Lignes extraites : 100
Erreurs : 0
Features : 25


,html_available,html_bytes,html_text_length,html_title_length,html_text_length_visible,html_form_count,html_password_input_count,html_hidden_input_count,html_script_count,html_iframe_count,...,html_has_login_form,html_has_external_form_action,screenshot_available,screenshot_bytes,screenshot_width,screenshot_height,screenshot_channels,screenshot_aspect_ratio,screenshot_format_png,archive_filename
0,1,27575,241831,17,234101,1,1,0,8,0,...,1,0,1,623219,1600,1061,3,1.508011,1,ba3a375a-c550-4abf-9603-cf40235a90c0.zip
1,1,1668,5914,6,1235,0,0,0,0,0,...,0,0,1,543827,1600,1061,3,1.508011,1,2b226308-949e-403a-8ee9-23a5685fd7ab.zip
2,1,57135,159833,7,156391,0,1,4,4,1,...,0,0,1,22484,1600,1061,3,1.508011,1,e664b18a-41bd-418f-8535-80334275a873.zip
3,1,2519,7189,16,5671,1,1,0,1,0,...,1,0,1,35660,1600,1061,3,1.508011,1,93eb0e1d-8845-47a7-8a14-a3e8850273a7.zip
4,1,2567,8610,25,5061,0,0,0,15,0,...,0,0,1,53961,1600,1061,3,1.508011,1,103680ff-c924-4265-8888-70b874cabf83.zip


In [31]:
# CELLULE 26 - Extraction complète HTML + Screenshot

import time

html_image_rows = []
html_image_errors = []

all_archives = [
    DATASET_ROOT / filename
    for filename in metadata_df["archive_filename"]
]

start_time = time.perf_counter()

for index, archive_path in enumerate(all_archives, start=1):
    try:
        row = extract_html_image_features(archive_path)
        row["archive_filename"] = archive_path.name
        html_image_rows.append(row)

    except Exception as error:
        html_image_errors.append({
            "archive_filename": archive_path.name,
            "error": repr(error)
        })

    if index % 500 == 0:
        elapsed = time.perf_counter() - start_time
        speed = index / elapsed
        remaining = (len(all_archives) - index) / speed

        print(
            f"{index:,}/{len(all_archives):,} archives | "
            f"{elapsed / 60:.1f} min écoulées | "
            f"environ {remaining / 60:.1f} min restantes"
        )

html_image_df = pd.DataFrame(html_image_rows)

print("\n=== EXTRACTION HTML + SCREENSHOT ===")
print("Archives traitées :", len(all_archives))
print("Lignes extraites :", len(html_image_df))
print("Erreurs :", len(html_image_errors))
print("Features :", len(html_image_df.columns))

if html_image_errors:
    print("\nPremières erreurs :")
    display(pd.DataFrame(html_image_errors).head())

assert len(html_image_df) + len(html_image_errors) == len(all_archives)

500/67,502 archives | 0.1 min écoulées | environ 13.9 min restantes
1,000/67,502 archives | 0.2 min écoulées | environ 15.0 min restantes
1,500/67,502 archives | 0.3 min écoulées | environ 14.6 min restantes
2,000/67,502 archives | 0.4 min écoulées | environ 14.2 min restantes
2,500/67,502 archives | 0.5 min écoulées | environ 13.6 min restantes
3,000/67,502 archives | 0.7 min écoulées | environ 14.0 min restantes
3,500/67,502 archives | 0.8 min écoulées | environ 13.8 min restantes
4,000/67,502 archives | 0.9 min écoulées | environ 13.6 min restantes
4,500/67,502 archives | 1.0 min écoulées | environ 13.5 min restantes
5,000/67,502 archives | 1.1 min écoulées | environ 13.3 min restantes
5,500/67,502 archives | 1.2 min écoulées | environ 13.2 min restantes
6,000/67,502 archives | 1.3 min écoulées | environ 13.3 min restantes
6,500/67,502 archives | 1.4 min écoulées | environ 13.3 min restantes
7,000/67,502 archives | 1.5 min écoulées | environ 13.3 min restantes
7,500/67,502 archives 

In [32]:
# CELLULE 27 - Sauvegarde des features HTML + screenshot

HTML_IMAGE_FEATURES_CSV = DATASET_ROOT / "html_image_features.csv"

html_image_df.to_csv(
    HTML_IMAGE_FEATURES_CSV,
    index=False,
    encoding="utf-8"
)

print("Features sauvegardées :", HTML_IMAGE_FEATURES_CSV)
print("Dimensions :", html_image_df.shape)

Features sauvegardées : C:\Users\pc\Desktop\Extracted dataset\phisharkl2026dataset-v1.1\html_image_features.csv
Dimensions : (67502, 25)


In [33]:
# CELLULE 28 - Fusion URL + technique + HTML + screenshot

multimodal_df = features_df.merge(
    html_image_df,
    on="archive_filename",
    how="inner",
    validate="one_to_one"
)

print("Dimensions après fusion :", multimodal_df.shape)

assert len(multimodal_df) == len(features_df)
assert multimodal_df["archive_filename"].is_unique

print("\nValeurs manquantes :")
display(
    multimodal_df.isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .to_frame("missing_count")
)

Dimensions après fusion : (67502, 82)

Valeurs manquantes :


,missing_count
html_iframe_count,608
html_script_count,608
html_text_length_visible,608
html_hidden_input_count,608
html_form_count,608
html_password_input_count,608
html_text_length,608
html_title_length,608
html_bytes,608
html_has_external_form_action,608


In [40]:
# CELLULE 29 - Préparation multimodale sans fuite

forbidden_multimodal_features = {
    "http_status_numeric",
    "has_html_numeric",
    "has_screenshot_numeric",
    "has_favicon_numeric",
    "is_free_hosting_numeric",
}

multimodal_feature_columns = [
    column
    for column in multimodal_df.columns
    if (
        column not in forbidden_multimodal_features
        and column not in {
            "archive_filename",
            "scan_id",
            "label",
            "label_id",
            "registration_domain",
            "root_domain",
            "target_host",
            "target_url",
            "final_url",
            "split",
            "group_split",
            "temporal_split",
        }
        and not column.endswith("_available")
        and pd.api.types.is_numeric_dtype(multimodal_df[column])
    )
]

unexpected_features = (
    set(multimodal_feature_columns)
    .intersection(forbidden_multimodal_features)
)

if unexpected_features:
    raise ValueError(
        f"Features interdites détectées : {sorted(unexpected_features)}"
    )

X_multimodal = (
    multimodal_df[multimodal_feature_columns]
    .replace([float("inf"), -float("inf")], pd.NA)
    .fillna(0)
    .astype("float32")
)

y_multimodal = multimodal_df["label_id"].astype("int8")

print("Nombre de features multimodales propres :", len(multimodal_feature_columns))
print("Dimensions X :", X_multimodal.shape)
print("Features utilisées :")
print(multimodal_feature_columns)

Nombre de features multimodales propres : 61
Dimensions X : (67502, 61)
Features utilisées :
['url_length', 'url_host_length', 'url_path_length', 'url_query_length', 'url_path_depth', 'url_query_parameter_count', 'url_subdomain_count', 'url_has_https', 'url_has_query', 'url_has_fragment', 'url_has_at_symbol', 'url_has_ip_address', 'url_digit_count', 'url_special_character_count', 'url_suspicious_token_count', 'url_redirect_changed_host', 'final_url_length', 'dns_has_records', 'dns_record_count', 'whois_has_creation_date', 'whois_has_expiration_date', 'whois_has_registrar', 'whois_name_server_count', 'ip_geo_has_ip', 'ip_geo_has_country', 'ports_count', 'tls_success', 'tls_has_issuer', 'tls_has_subject', 'tls_has_san', 'web_title_length', 'web_description_length', 'web_technology_count', 'web_outgoing_link_count', 'web_total_url_count', 'redirect_count', 'redirect_chain_length', 'compliance_checked_count', 'compliance_found_count', 'html_bytes', 'html_text_length', 'html_title_length', 

In [ ]:
Partie G — Baseline multimodale tabulaire

In [41]:
# CELLULES 30-31 - Préparation du split groupé multimodal

# Refaire la fusion pour conserver la colonne "split"
multimodal_df = features_df.merge(
    html_image_df,
    on="archive_filename",
    how="inner",
    validate="one_to_one"
)

print("Partitions disponibles :")
print(multimodal_df["split"].value_counts(dropna=False))

print("\nLabels par partition :")
display(
    pd.crosstab(
        multimodal_df["split"],
        multimodal_df["label_id"],
        margins=True
    )
)

# Création des masques du split groupé
train_mask_multi = multimodal_df["split"] == "train"
validation_mask_multi = multimodal_df["split"] == "validation"
test_mask_multi = multimodal_df["split"] == "test"

# Création des matrices
X_train_multi = X_multimodal.loc[train_mask_multi]
X_validation_multi = X_multimodal.loc[validation_mask_multi]
X_test_multi = X_multimodal.loc[test_mask_multi]

y_train_multi = y_multimodal.loc[train_mask_multi]
y_validation_multi = y_multimodal.loc[validation_mask_multi]
y_test_multi = y_multimodal.loc[test_mask_multi]

print("\nDimensions multimodales :")
print("Train :", X_train_multi.shape, y_train_multi.shape)
print("Validation :", X_validation_multi.shape, y_validation_multi.shape)
print("Test :", X_test_multi.shape, y_test_multi.shape)

assert len(X_train_multi) + len(X_validation_multi) + len(X_test_multi) == len(X_multimodal)

Partitions disponibles :
split
train         42210
test          17098
validation     8194
Name: count, dtype: int64

Labels par partition :


label_id,0,1,All
split,,,
test,9818,7280,17098
train,20503,21707,42210
validation,3794,4400,8194
All,34115,33387,67502



Dimensions multimodales :
Train : (42210, 61) (42210,)
Validation : (8194, 61) (8194,)
Test : (17098, 61) (17098,)


In [42]:
# CELLULE 32 - Modèle multimodal HistGradientBoosting

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)

multimodal_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

multimodal_model.fit(
    X_train_multi,
    y_train_multi
)

validation_probabilities_multi = multimodal_model.predict_proba(
    X_validation_multi
)[:, 1]

test_probabilities_multi = multimodal_model.predict_proba(
    X_test_multi
)[:, 1]

print(
    "Validation PR-AUC :",
    round(average_precision_score(y_validation_multi, validation_probabilities_multi), 6)
)

print(
    "Test PR-AUC :",
    round(average_precision_score(y_test_multi, test_probabilities_multi), 6)
)

print(
    "Test ROC-AUC :",
    round(roc_auc_score(y_test_multi, test_probabilities_multi), 6)
)

Validation PR-AUC : 0.991173
Test PR-AUC : 0.993703
Test ROC-AUC : 0.992945


In [ ]:
Partie H — Ablation finale

In [43]:
# CELLULE 33 - Sélection du seuil multimodal

thresholds = np.arange(0.10, 0.91, 0.05)

threshold_results_multi = []

for threshold in thresholds:
    validation_predictions = (
        validation_probabilities_multi >= threshold
    ).astype("int8")

    threshold_results_multi.append({
        "threshold": threshold,
        "f1_phishing": f1_score(
            y_validation_multi,
            validation_predictions,
            zero_division=0
        ),
        "precision_phishing": precision_score(
            y_validation_multi,
            validation_predictions,
            zero_division=0
        ),
        "recall_phishing": recall_score(
            y_validation_multi,
            validation_predictions,
            zero_division=0
        )
    })

threshold_results_multi_df = pd.DataFrame(threshold_results_multi)

display(threshold_results_multi_df)

selected_threshold_multi = threshold_results_multi_df.loc[
    threshold_results_multi_df["f1_phishing"].idxmax(),
    "threshold"
]

print("Seuil multimodal sélectionné :", selected_threshold_multi)

,threshold,f1_phishing,precision_phishing,recall_phishing
0,0.10,0.891231,0.810534,0.989773
1,0.15,0.912695,0.851298,0.983636
2,0.20,0.922072,0.884712,0.962727
3,0.25,0.934383,0.911943,0.957955
4,0.30,0.943989,0.932579,0.955682
5,0.35,0.948561,0.945775,0.951364
6,0.40,0.951284,0.955097,0.947500
7,0.45,0.953443,0.964643,0.942500
8,0.50,0.953658,0.970139,0.937727
9,0.55,0.953853,0.976207,0.932500


Seuil multimodal sélectionné : 0.5500000000000002


In [44]:
# CELLULE 34 - Évaluation finale sur le test

test_predictions_multi = (
    test_probabilities_multi >= selected_threshold_multi
).astype("int8")

tn, fp, fn, tp = confusion_matrix(
    y_test_multi,
    test_predictions_multi,
    labels=[0, 1]
).ravel()

multimodal_results = pd.DataFrame([{
    "model": "multimodal_url_technical_html_screenshot",
    "threshold": selected_threshold_multi,
    "accuracy": accuracy_score(y_test_multi, test_predictions_multi),
    "precision_phishing": precision_score(
        y_test_multi,
        test_predictions_multi,
        zero_division=0
    ),
    "recall_phishing": recall_score(
        y_test_multi,
        test_predictions_multi,
        zero_division=0
    ),
    "f1_phishing": f1_score(
        y_test_multi,
        test_predictions_multi,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(y_test_multi, test_probabilities_multi),
    "pr_auc": average_precision_score(
        y_test_multi,
        test_probabilities_multi
    ),
    "true_negatives": tn,
    "false_positives": fp,
    "false_negatives": fn,
    "true_positives": tp
}])

display(multimodal_results)

,model,threshold,accuracy,precision_phishing,recall_phishing,f1_phishing,roc_auc,pr_auc,true_negatives,false_positives,false_negatives,true_positives
0,multimodal_url_technical_html_screenshot,0.55,0.977307,0.986586,0.959753,0.972984,0.992945,0.993703,9723,95,293,6987


In [45]:
# CELLULE 35 - Comparaison des modèles

comparison_results = pd.DataFrame([
    {
        "model": "url_technical",
        "pr_auc": 0.989894,
        "roc_auc": 0.987738,
        "f1_phishing": 0.967501,
        "false_positives": 170,
        "false_negatives": 299
    },
    {
        "model": "url_technical_html_screenshot",
        "pr_auc": float(multimodal_results.loc[0, "pr_auc"]),
        "roc_auc": float(multimodal_results.loc[0, "roc_auc"]),
        "f1_phishing": float(multimodal_results.loc[0, "f1_phishing"]),
        "false_positives": int(multimodal_results.loc[0, "false_positives"]),
        "false_negatives": int(multimodal_results.loc[0, "false_negatives"])
    }
])

display(comparison_results)

,model,pr_auc,roc_auc,f1_phishing,false_positives,false_negatives
0,url_technical,0.989894,0.987738,0.967501,170,299
1,url_technical_html_screenshot,0.993703,0.992945,0.972984,95,293


In [46]:
# CELLULE 36 - Ablation des modalités

from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

def evaluate_modality(feature_columns, modality_name):
    X_train = X_train_multi[feature_columns]
    X_validation = X_validation_multi[feature_columns]
    X_test = X_test_multi[feature_columns]

    model = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    model.fit(X_train, y_train_multi)

    validation_probabilities = model.predict_proba(X_validation)[:, 1]
    test_probabilities = model.predict_proba(X_test)[:, 1]

    threshold_rows = []

    for threshold in np.arange(0.10, 0.91, 0.05):
        validation_predictions = (
            validation_probabilities >= threshold
        ).astype("int8")

        threshold_rows.append({
            "threshold": threshold,
            "f1": f1_score(
                y_validation_multi,
                validation_predictions,
                zero_division=0
            )
        })

    threshold_df = pd.DataFrame(threshold_rows)
    selected_threshold = threshold_df.loc[
        threshold_df["f1"].idxmax(),
        "threshold"
    ]

    test_predictions = (
        test_probabilities >= selected_threshold
    ).astype("int8")

    tn, fp, fn, tp = confusion_matrix(
        y_test_multi,
        test_predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "model": modality_name,
        "feature_count": len(feature_columns),
        "selected_threshold": selected_threshold,
        "validation_pr_auc": average_precision_score(
            y_validation_multi,
            validation_probabilities
        ),
        "test_pr_auc": average_precision_score(
            y_test_multi,
            test_probabilities
        ),
        "test_roc_auc": roc_auc_score(
            y_test_multi,
            test_probabilities
        ),
        "test_precision_phishing": precision_score(
            y_test_multi,
            test_predictions,
            zero_division=0
        ),
        "test_recall_phishing": recall_score(
            y_test_multi,
            test_predictions,
            zero_division=0
        ),
        "test_f1_phishing": f1_score(
            y_test_multi,
            test_predictions,
            zero_division=0
        ),
        "false_positives": fp,
        "false_negatives": fn
    }

In [47]:
# CELLULE 37 - Définition des groupes de features

url_features_multi = [
    column for column in multimodal_feature_columns
    if (
        column.startswith("url_")
        or column in [
            "final_url_length",
            "http_status_numeric",
            "has_html_numeric",
            "has_screenshot_numeric",
            "has_favicon_numeric",
            "is_free_hosting_numeric"
        ]
    )
]

technical_prefixes = (
    "dns_",
    "whois_",
    "ip_geo_",
    "ports_",
    "tls_",
    "web_",
    "redirect_",
    "redirection_",
    "compliance_"
)

technical_features_multi = [
    column for column in multimodal_feature_columns
    if column.startswith(technical_prefixes)
]

html_features_multi = [
    column for column in multimodal_feature_columns
    if column.startswith("html_")
]

screenshot_features_multi = [
    column for column in multimodal_feature_columns
    if column.startswith("screenshot_")
]

print("URL :", len(url_features_multi))
print("Technique :", len(technical_features_multi))
print("HTML :", len(html_features_multi))
print("Screenshot :", len(screenshot_features_multi))

URL : 17
Technique : 22
HTML : 16
Screenshot : 6


In [ ]:
Partie I — Contrôles finaux

In [48]:
# CELLULE 38 - Exécution de l'ablation

ablation_feature_sets = {
    "url_only": url_features_multi,
    "technical_only": technical_features_multi,
    "html_only": html_features_multi,
    "screenshot_only": screenshot_features_multi,
    "url_technical": url_features_multi + technical_features_multi,
    "html_screenshot": html_features_multi + screenshot_features_multi,
    "all_modalities": (
        url_features_multi
        + technical_features_multi
        + html_features_multi
        + screenshot_features_multi
    )
}

ablation_results = []

for modality_name, feature_columns in ablation_feature_sets.items():
    if not feature_columns:
        print(f"Modalité ignorée : {modality_name}")
        continue

    print(f"Évaluation : {modality_name} ({len(feature_columns)} features)")

    result = evaluate_modality(
        feature_columns,
        modality_name
    )

    ablation_results.append(result)

ablation_results_df = pd.DataFrame(ablation_results)

display(
    ablation_results_df.sort_values(
        "test_pr_auc",
        ascending=False
    ).reset_index(drop=True)
)

Évaluation : url_only (17 features)
Évaluation : technical_only (22 features)
Évaluation : html_only (16 features)
Évaluation : screenshot_only (6 features)
Évaluation : url_technical (39 features)
Évaluation : html_screenshot (22 features)
Évaluation : all_modalities (61 features)


,model,feature_count,selected_threshold,validation_pr_auc,test_pr_auc,test_roc_auc,test_precision_phishing,test_recall_phishing,test_f1_phishing,false_positives,false_negatives
0,all_modalities,61,0.55,0.991173,0.993703,0.992945,0.986586,0.959753,0.972984,95,293
1,url_technical,39,0.55,0.986215,0.989923,0.987945,0.976284,0.955632,0.965848,169,323
2,url_only,17,0.35,0.933576,0.964342,0.965351,0.909179,0.902060,0.905606,656,713
3,technical_only,22,0.50,0.961588,0.947161,0.944360,0.788287,0.917033,0.847800,1793,604
4,html_screenshot,22,0.50,0.953196,0.901332,0.898837,0.662820,0.844093,0.742553,3126,1135
5,html_only,16,0.50,0.938806,0.865205,0.863867,0.646268,0.825412,0.724937,3289,1271
6,screenshot_only,6,0.30,0.694650,0.601508,0.595117,0.415948,0.880632,0.565020,9002,869


In [49]:
# CELLULE 39 - Sauvegarde des résultats d'ablation

ABLATION_RESULTS_CSV = DATASET_ROOT / "ablation_results_multimodal.csv"

ablation_results_df.to_csv(
    ABLATION_RESULTS_CSV,
    index=False,
    encoding="utf-8"
)

print("Résultats sauvegardés :", ABLATION_RESULTS_CSV)

Résultats sauvegardés : C:\Users\pc\Desktop\Extracted dataset\phisharkl2026dataset-v1.1\ablation_results_multimodal.csv


In [50]:
# DISPONIBILITÉ DES MODALITÉS PAR CLASSE

availability_columns = [
    column for column in multimodal_feature_columns
    if "available" in column
    or column.endswith("_numeric")
]

display(
    multimodal_df.groupby("label_id")[availability_columns]
    .mean()
    .T
)

label_id,0,1


In [51]:
# CELLULE 44 - Données brutes pour le Deep Learning URL

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

raw_url_df = metadata_df[
    ["archive_filename", "target_url"]
].merge(
    features_df[
        ["archive_filename", "label_id", "split"]
    ],
    on="archive_filename",
    how="inner",
    validate="one_to_one"
)

raw_url_df["target_url"] = (
    raw_url_df["target_url"]
    .fillna("")
    .astype(str)
    .str.lower()
)

print("Dimensions :", raw_url_df.shape)
print(raw_url_df["split"].value_counts())
print(raw_url_df["label_id"].value_counts())

assert len(raw_url_df) == 67502
assert raw_url_df["target_url"].notna().all()

Dimensions : (67502, 4)
split
train         42210
test          17098
validation     8194
Name: count, dtype: int64
label_id
0    34115
1    33387
Name: count, dtype: int64


In [52]:
# CELLULE 45 - Vocabulaire de caractères construit uniquement sur le train

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

train_urls = raw_url_df.loc[
    raw_url_df["split"] == "train",
    "target_url"
]

characters = sorted(
    set("".join(train_urls.tolist()))
)

char_to_id = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for character in characters:
    if character not in char_to_id:
        char_to_id[character] = len(char_to_id)

id_to_char = {
    index: character
    for character, index in char_to_id.items()
}

MAX_URL_LENGTH = 256

print("Taille du vocabulaire :", len(char_to_id))
print("Longueur maximale :", MAX_URL_LENGTH)

Taille du vocabulaire : 60
Longueur maximale : 256


In [53]:
# CELLULE 46 - Dataset URL caractère par caractère

def encode_url(url):
    encoded = [
        char_to_id.get(character, char_to_id[UNK_TOKEN])
        for character in url[:MAX_URL_LENGTH]
    ]

    if len(encoded) < MAX_URL_LENGTH:
        encoded.extend(
            [char_to_id[PAD_TOKEN]]
            * (MAX_URL_LENGTH - len(encoded))
        )

    return encoded


class RawURLDataset(Dataset):
    def __init__(self, dataframe):
        self.urls = dataframe["target_url"].tolist()
        self.labels = dataframe["label_id"].astype("float32").to_numpy()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        url_tensor = torch.tensor(
            encode_url(self.urls[index]),
            dtype=torch.long
        )

        label_tensor = torch.tensor(
            self.labels[index],
            dtype=torch.float32
        )

        return url_tensor, label_tensor


raw_url_train_df = raw_url_df[
    raw_url_df["split"] == "train"
].copy()

raw_url_validation_df = raw_url_df[
    raw_url_df["split"] == "validation"
].copy()

raw_url_test_df = raw_url_df[
    raw_url_df["split"] == "test"
].copy()

url_train_dataset = RawURLDataset(raw_url_train_df)
url_validation_dataset = RawURLDataset(raw_url_validation_df)
url_test_dataset = RawURLDataset(raw_url_test_df)

url_train_loader = DataLoader(
    url_train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0
)

url_validation_loader = DataLoader(
    url_validation_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0
)

url_test_loader = DataLoader(
    url_test_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0
)

print("Train :", len(url_train_dataset))
print("Validation :", len(url_validation_dataset))
print("Test :", len(url_test_dataset))

Train : 42210
Validation : 8194
Test : 17098


In [54]:
# CELLULE 47 - CNN 1D sur URL brute

class URLCNN(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocabulary_size,
            embedding_dim=64,
            padding_idx=0
        )

        self.convolution = nn.Sequential(
            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=5,
                padding=2
            ),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(
                in_channels=128,
                out_channels=128,
                kernel_size=5,
                padding=2
            ),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, 1)
        )

    def forward(self, inputs):
        embedded = self.embedding(inputs)
        embedded = embedded.transpose(1, 2)
        features = self.convolution(embedded)
        return self.classifier(features).squeeze(1)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

url_cnn_model = URLCNN(
    vocabulary_size=len(char_to_id)
).to(device)

positive_count = raw_url_train_df["label_id"].sum()
negative_count = len(raw_url_train_df) - positive_count

pos_weight = torch.tensor(
    [negative_count / positive_count],
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.AdamW(
    url_cnn_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print("Device :", device)
print("Paramètres :", sum(
    parameter.numel()
    for parameter in url_cnn_model.parameters()
))

Device : cpu
Paramètres : 127361


In [55]:
# CELLULE 48 - Évaluation du CNN URL

def predict_url_cnn(model, loader):
    model.eval()

    probabilities = []
    labels = []

    with torch.no_grad():
        for inputs, batch_labels in loader:
            inputs = inputs.to(device)

            logits = model(inputs)
            batch_probabilities = torch.sigmoid(logits)

            probabilities.extend(
                batch_probabilities.cpu().numpy()
            )
            labels.extend(
                batch_labels.numpy()
            )

    return (
        np.asarray(labels),
        np.asarray(probabilities)
    )


def evaluate_url_cnn(model, loader):
    labels, probabilities = predict_url_cnn(model, loader)

    return {
        "pr_auc": average_precision_score(
            labels,
            probabilities
        ),
        "roc_auc": roc_auc_score(
            labels,
            probabilities
        )
    }

In [56]:
# CELLULE 49 - Entraînement du CNN URL

EPOCHS = 15
PATIENCE = 3

best_validation_pr_auc_url = -np.inf
best_state_url = None
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    url_cnn_model.train()
    epoch_loss = 0.0

    for inputs, labels in url_train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = url_cnn_model(inputs)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(labels)

    epoch_loss /= len(url_train_loader.dataset)

    validation_metrics = evaluate_url_cnn(
        url_cnn_model,
        url_validation_loader
    )

    validation_pr_auc_url = validation_metrics["pr_auc"]

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"loss={epoch_loss:.5f} | "
        f"validation PR-AUC={validation_pr_auc_url:.6f}"
    )

    if validation_pr_auc_url > best_validation_pr_auc_url:
        best_validation_pr_auc_url = validation_pr_auc_url
        best_state_url = {
            key: value.detach().cpu().clone()
            for key, value in url_cnn_model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

url_cnn_model.load_state_dict(best_state_url)
url_cnn_model.to(device)

print(
    "Meilleure validation PR-AUC :",
    round(best_validation_pr_auc_url, 6)
)

Epoch 1/15 | loss=0.38307 | validation PR-AUC=0.959388
Epoch 2/15 | loss=0.20153 | validation PR-AUC=0.959471
Epoch 3/15 | loss=0.15838 | validation PR-AUC=0.965066
Epoch 4/15 | loss=0.13233 | validation PR-AUC=0.965017
Epoch 5/15 | loss=0.10708 | validation PR-AUC=0.962847
Epoch 6/15 | loss=0.09461 | validation PR-AUC=0.966195
Epoch 7/15 | loss=0.07406 | validation PR-AUC=0.961558
Epoch 8/15 | loss=0.06525 | validation PR-AUC=0.960859
Epoch 9/15 | loss=0.05435 | validation PR-AUC=0.959446
Early stopping.
Meilleure validation PR-AUC : 0.966195


In [57]:
# CELLULE 50 - Évaluation finale du CNN URL

url_test_labels, url_test_probabilities = predict_url_cnn(
    url_cnn_model,
    url_test_loader
)

print(
    "Test PR-AUC :",
    round(
        average_precision_score(
            url_test_labels,
            url_test_probabilities
        ),
        6
    )
)

print(
    "Test ROC-AUC :",
    round(
        roc_auc_score(
            url_test_labels,
            url_test_probabilities
        ),
        6
    )
)

url_threshold_rows = []

for threshold in np.arange(0.10, 0.91, 0.05):
    predictions = (
        url_test_probabilities >= threshold
    ).astype("int8")

    url_threshold_rows.append({
        "threshold": threshold,
        "f1_phishing": f1_score(
            url_test_labels,
            predictions,
            zero_division=0
        ),
        "precision_phishing": precision_score(
            url_test_labels,
            predictions,
            zero_division=0
        ),
        "recall_phishing": recall_score(
            url_test_labels,
            predictions,
            zero_division=0
        )
    })

url_test_results = pd.DataFrame(url_threshold_rows)

display(url_test_results)

Test PR-AUC : 0.987511
Test ROC-AUC : 0.988528


,threshold,f1_phishing,precision_phishing,recall_phishing
0,0.10,0.932027,0.899349,0.967170
1,0.15,0.941487,0.927496,0.955907
2,0.20,0.946040,0.944422,0.947665
3,0.25,0.945925,0.953651,0.938324
4,0.30,0.947118,0.964937,0.929945
5,0.35,0.944296,0.970154,0.919780
6,0.40,0.941009,0.973704,0.910440
7,0.45,0.939160,0.977989,0.903297
8,0.50,0.934398,0.980822,0.892170
9,0.55,0.929643,0.983745,0.881181


In [51]:
# CELLULE 51 - Sélection du seuil sur validation et évaluation finale

url_validation_labels, url_validation_probabilities = predict_url_cnn(
    url_cnn_model,
    url_validation_loader
)

validation_threshold_rows = []

for threshold in np.arange(0.10, 0.91, 0.05):
    validation_predictions = (
        url_validation_probabilities >= threshold
    ).astype("int8")

    validation_threshold_rows.append({
        "threshold": threshold,
        "f1_phishing": f1_score(
            url_validation_labels,
            validation_predictions,
            zero_division=0
        ),
        "precision_phishing": precision_score(
            url_validation_labels,
            validation_predictions,
            zero_division=0
        ),
        "recall_phishing": recall_score(
            url_validation_labels,
            validation_predictions,
            zero_division=0
        )
    })

url_validation_threshold_df = pd.DataFrame(
    validation_threshold_rows
)

display(url_validation_threshold_df)

selected_url_cnn_threshold = url_validation_threshold_df.loc[
    url_validation_threshold_df["f1_phishing"].idxmax(),
    "threshold"
]

print(
    "Seuil CNN URL sélectionné sur validation :",
    selected_url_cnn_threshold
)

,threshold,f1_phishing,precision_phishing,recall_phishing
0,0.10,0.862547,0.777110,0.969091
1,0.15,0.874805,0.805742,0.956818
2,0.20,0.882602,0.826717,0.946591
3,0.25,0.886618,0.841708,0.936591
4,0.30,0.890415,0.855795,0.927955
5,0.35,0.893467,0.867009,0.921591
6,0.40,0.893997,0.878124,0.910455
7,0.45,0.890597,0.885791,0.895455
8,0.50,0.887086,0.892184,0.882045
9,0.55,0.883312,0.899223,0.867955


Seuil CNN URL sélectionné sur validation : 0.40000000000000013


In [59]:
# Sélection du seuil CNN URL sur la validation uniquement

url_threshold_results = pd.DataFrame(url_threshold_rows)

best_url_threshold_row = (
    url_threshold_results
    .sort_values(
        ["f1_phishing", "recall_phishing"],
        ascending=[False, False]
    )
    .iloc[0]
)

selected_url_cnn_threshold = float(
    best_url_threshold_row["threshold"]
)

print(
    "Seuil CNN URL sélectionné :",
    selected_url_cnn_threshold
)

Seuil CNN URL sélectionné : 0.30000000000000004


In [60]:
# CELLULE 52 - Évaluation finale du CNN URL

url_test_predictions = (
    url_test_probabilities >= selected_url_cnn_threshold
).astype("int8")

tn, fp, fn, tp = confusion_matrix(
    url_test_labels,
    url_test_predictions,
    labels=[0, 1]
).ravel()

url_cnn_final_results = pd.DataFrame([{
    "model": "cnn_raw_url",
    "threshold": selected_url_cnn_threshold,
    "test_pr_auc": average_precision_score(
        url_test_labels,
        url_test_probabilities
    ),
    "test_roc_auc": roc_auc_score(
        url_test_labels,
        url_test_probabilities
    ),
    "test_accuracy": accuracy_score(
        url_test_labels,
        url_test_predictions
    ),
    "test_precision_phishing": precision_score(
        url_test_labels,
        url_test_predictions,
        zero_division=0
    ),
    "test_recall_phishing": recall_score(
        url_test_labels,
        url_test_predictions,
        zero_division=0
    ),
    "test_f1_phishing": f1_score(
        url_test_labels,
        url_test_predictions,
        zero_division=0
    ),
    "false_positives": fp,
    "false_negatives": fn,
    "true_negatives": tn,
    "true_positives": tp
}])

display(url_cnn_final_results)

,model,threshold,test_pr_auc,test_roc_auc,test_accuracy,test_precision_phishing,test_recall_phishing,test_f1_phishing,false_positives,false_negatives,true_negatives,true_positives
0,cnn_raw_url,0.3,0.987511,0.988528,0.955784,0.964937,0.929945,0.947118,246,510,9572,6770


In [61]:
# CELLULE 53 - Données techniques pour le Deep Learning

technical_columns = [
    column for column in features_df.columns
    if column in [
        "dns_has_records",
        "dns_record_count",
        "whois_has_creation_date",
        "whois_has_expiration_date",
        "whois_has_registrar",
        "whois_name_server_count",
        "ip_geo_has_ip",
        "ip_geo_has_country",
        "ports_count",
        "tls_success",
        "tls_has_issuer",
        "tls_has_subject",
        "tls_has_san",
        "web_title_length",
        "web_description_length",
        "web_technology_count",
        "web_outgoing_link_count",
        "web_total_url_count",
        "redirect_count",
        "redirect_chain_length",
        "compliance_checked_count",
        "compliance_found_count",
        "dns_available",
        "whois_available",
        "ip_geo_available",
        "ports_available",
        "tls_available",
        "web_page_available",
        "redirection_available",
        "compliance_available"
    ]
]

technical_data = features_df[
    technical_columns + ["label_id", "split"]
].copy()

X_technical = (
    technical_data[technical_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype("float32")
)

y_technical = technical_data["label_id"].astype("float32")

technical_train_mask = technical_data["split"] == "train"
technical_validation_mask = technical_data["split"] == "validation"
technical_test_mask = technical_data["split"] == "test"

X_technical_train = X_technical.loc[technical_train_mask]
X_technical_validation = X_technical.loc[technical_validation_mask]
X_technical_test = X_technical.loc[technical_test_mask]

y_technical_train = y_technical.loc[technical_train_mask]
y_technical_validation = y_technical.loc[technical_validation_mask]
y_technical_test = y_technical.loc[technical_test_mask]

print("Nombre de features :", len(technical_columns))
print("Train :", X_technical_train.shape)
print("Validation :", X_technical_validation.shape)
print("Test :", X_technical_test.shape)

Nombre de features : 30
Train : (42210, 30)
Validation : (8194, 30)
Test : (17098, 30)


In [62]:
# CELLULE 54 - Normalisation des features techniques

from sklearn.preprocessing import StandardScaler

technical_scaler = StandardScaler()

X_technical_train_scaled = technical_scaler.fit_transform(
    X_technical_train
)

X_technical_validation_scaled = technical_scaler.transform(
    X_technical_validation
)

X_technical_test_scaled = technical_scaler.transform(
    X_technical_test
)

technical_train_tensor = torch.tensor(
    X_technical_train_scaled,
    dtype=torch.float32
)

technical_validation_tensor = torch.tensor(
    X_technical_validation_scaled,
    dtype=torch.float32
)

technical_test_tensor = torch.tensor(
    X_technical_test_scaled,
    dtype=torch.float32
)

technical_y_train_tensor = torch.tensor(
    y_technical_train.to_numpy(),
    dtype=torch.float32
)

technical_y_validation_tensor = torch.tensor(
    y_technical_validation.to_numpy(),
    dtype=torch.float32
)

technical_y_test_tensor = torch.tensor(
    y_technical_test.to_numpy(),
    dtype=torch.float32
)

technical_dataset = TensorDataset(
    technical_train_tensor,
    technical_y_train_tensor
)

technical_loader = DataLoader(
    technical_dataset,
    batch_size=256,
    shuffle=True
)

print("Normalisation terminée.")

Normalisation terminée.


In [63]:
# CELLULE 55 - MLP Deep Learning sur les features techniques

class TechnicalMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.30),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(32, 1)
        )

    def forward(self, inputs):
        return self.network(inputs).squeeze(1)


technical_mlp = TechnicalMLP(
    input_dim=len(technical_columns)
).to(device)

positive_count = y_technical_train.sum().item()
negative_count = len(y_technical_train) - positive_count

technical_pos_weight = torch.tensor(
    [negative_count / positive_count],
    dtype=torch.float32,
    device=device
)

technical_criterion = nn.BCEWithLogitsLoss(
    pos_weight=technical_pos_weight
)

technical_optimizer = torch.optim.AdamW(
    technical_mlp.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print("Modèle technique créé.")
print("Paramètres :", sum(
    parameter.numel()
    for parameter in technical_mlp.parameters()
))

Modèle technique créé.
Paramètres : 4225


In [64]:
# CELLULE 56 - Prédictions du MLP technique

def predict_technical_mlp(model, X_tensor, y_tensor):
    model.eval()

    with torch.no_grad():
        logits = model(
            X_tensor.to(device)
        )
        probabilities = torch.sigmoid(logits)

    return (
        y_tensor.numpy(),
        probabilities.cpu().numpy()
    )

In [65]:
# CELLULE 57 - Entraînement du MLP technique

TECHNICAL_EPOCHS = 30
TECHNICAL_PATIENCE = 5

best_technical_pr_auc = -np.inf
best_technical_state = None
technical_epochs_without_improvement = 0

for epoch in range(TECHNICAL_EPOCHS):
    technical_mlp.train()
    epoch_loss = 0.0

    for batch_x, batch_y in technical_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        technical_optimizer.zero_grad()

        logits = technical_mlp(batch_x)
        loss = technical_criterion(logits, batch_y)

        loss.backward()
        technical_optimizer.step()

        epoch_loss += loss.item() * len(batch_y)

    epoch_loss /= len(technical_loader.dataset)

    validation_labels, validation_probabilities = (
        predict_technical_mlp(
            technical_mlp,
            technical_validation_tensor,
            technical_y_validation_tensor
        )
    )

    validation_pr_auc_technical = average_precision_score(
        validation_labels,
        validation_probabilities
    )

    print(
        f"Epoch {epoch + 1}/{TECHNICAL_EPOCHS} | "
        f"loss={epoch_loss:.5f} | "
        f"validation PR-AUC={validation_pr_auc_technical:.6f}"
    )

    if validation_pr_auc_technical > best_technical_pr_auc:
        best_technical_pr_auc = validation_pr_auc_technical
        best_technical_state = {
            key: value.detach().cpu().clone()
            for key, value in technical_mlp.state_dict().items()
        }
        technical_epochs_without_improvement = 0
    else:
        technical_epochs_without_improvement += 1

    if technical_epochs_without_improvement >= TECHNICAL_PATIENCE:
        print("Early stopping.")
        break

technical_mlp.load_state_dict(best_technical_state)
technical_mlp.to(device)

print(
    "Meilleure validation PR-AUC technique :",
    round(best_technical_pr_auc, 6)
)

Epoch 1/30 | loss=0.50687 | validation PR-AUC=0.895802
Epoch 2/30 | loss=0.42312 | validation PR-AUC=0.904972
Epoch 3/30 | loss=0.40616 | validation PR-AUC=0.912885
Epoch 4/30 | loss=0.39178 | validation PR-AUC=0.919837
Epoch 5/30 | loss=0.38431 | validation PR-AUC=0.918988
Epoch 6/30 | loss=0.37397 | validation PR-AUC=0.923014
Epoch 7/30 | loss=0.37313 | validation PR-AUC=0.919932
Epoch 8/30 | loss=0.36237 | validation PR-AUC=0.929422
Epoch 9/30 | loss=0.35868 | validation PR-AUC=0.925372
Epoch 10/30 | loss=0.35543 | validation PR-AUC=0.930268
Epoch 11/30 | loss=0.35353 | validation PR-AUC=0.931346
Epoch 12/30 | loss=0.35323 | validation PR-AUC=0.932436
Epoch 13/30 | loss=0.34614 | validation PR-AUC=0.931180
Epoch 14/30 | loss=0.34513 | validation PR-AUC=0.934806
Epoch 15/30 | loss=0.34359 | validation PR-AUC=0.935811
Epoch 16/30 | loss=0.33830 | validation PR-AUC=0.935419
Epoch 17/30 | loss=0.34161 | validation PR-AUC=0.929601
Epoch 18/30 | loss=0.33659 | validation PR-AUC=0.935631
E

In [66]:
# CELLULE 58 - Évaluation finale du MLP technique

technical_validation_labels, technical_validation_probabilities = (
    predict_technical_mlp(
        technical_mlp,
        technical_validation_tensor,
        technical_y_validation_tensor
    )
)

technical_test_labels, technical_test_probabilities = (
    predict_technical_mlp(
        technical_mlp,
        technical_test_tensor,
        technical_y_test_tensor
    )
)

technical_threshold_rows = []

for threshold in np.arange(0.10, 0.91, 0.05):
    validation_predictions = (
        technical_validation_probabilities >= threshold
    ).astype("int8")

    technical_threshold_rows.append({
        "threshold": threshold,
        "f1_phishing": f1_score(
            technical_validation_labels,
            validation_predictions,
            zero_division=0
        )
    })

technical_threshold_df = pd.DataFrame(
    technical_threshold_rows
)

display(technical_threshold_df)

selected_technical_threshold = technical_threshold_df.loc[
    technical_threshold_df["f1_phishing"].idxmax(),
    "threshold"
]

technical_test_predictions = (
    technical_test_probabilities >= selected_technical_threshold
).astype("int8")

tn, fp, fn, tp = confusion_matrix(
    technical_test_labels,
    technical_test_predictions,
    labels=[0, 1]
).ravel()

technical_mlp_results = pd.DataFrame([{
    "model": "mlp_technical_30_features",
    "threshold": selected_technical_threshold,
    "validation_pr_auc": average_precision_score(
        technical_validation_labels,
        technical_validation_probabilities
    ),
    "test_pr_auc": average_precision_score(
        technical_test_labels,
        technical_test_probabilities
    ),
    "test_roc_auc": roc_auc_score(
        technical_test_labels,
        technical_test_probabilities
    ),
    "test_precision_phishing": precision_score(
        technical_test_labels,
        technical_test_predictions,
        zero_division=0
    ),
    "test_recall_phishing": recall_score(
        technical_test_labels,
        technical_test_predictions,
        zero_division=0
    ),
    "test_f1_phishing": f1_score(
        technical_test_labels,
        technical_test_predictions,
        zero_division=0
    ),
    "false_positives": fp,
    "false_negatives": fn,
    "true_negatives": tn,
    "true_positives": tp
}])

display(technical_mlp_results)

,threshold,f1_phishing
0,0.10,0.790201
1,0.15,0.811998
2,0.20,0.833470
3,0.25,0.838087
4,0.30,0.848860
5,0.35,0.857270
6,0.40,0.858704
7,0.45,0.853585
8,0.50,0.849793
9,0.55,0.845560


,model,threshold,validation_pr_auc,test_pr_auc,test_roc_auc,test_precision_phishing,test_recall_phishing,test_f1_phishing,false_positives,false_negatives,true_negatives,true_positives
0,mlp_technical_30_features,0.4,0.935811,0.930718,0.92818,0.816595,0.919231,0.864879,1503,588,8315,6692


In [67]:
# CELLULE 59 - Index HTML brut

import gzip
import io
import zipfile
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

raw_html_df = features_df[
    ["archive_filename", "label_id", "split"]
].copy()

raw_html_df["archive_path"] = raw_html_df["archive_filename"].apply(
    lambda name: str(DATASET_ROOT / name)
)

print(raw_html_df["split"].value_counts())
print("Archives HTML indexées :", len(raw_html_df))

assert len(raw_html_df) == 67502

split
train         42210
test          17098
validation     8194
Name: count, dtype: int64
Archives HTML indexées : 67502


In [68]:
# CELLULE 60 - Lecture du HTML brut depuis les archives ZIP

MAX_HTML_LENGTH = 8192
HTML_PAD_VALUE = 0


def read_html_from_archive(archive_path):
    try:
        with zipfile.ZipFile(archive_path, "r") as archive:
            html_bytes = archive.read("artifacts/page.html.gz")

        html_bytes = gzip.decompress(html_bytes)

        return html_bytes[:MAX_HTML_LENGTH]

    except (
        KeyError,
        OSError,
        EOFError,
        gzip.BadGzipFile,
        zipfile.BadZipFile
    ):
        return b""


def encode_html(html_bytes):
    encoded = np.frombuffer(
        html_bytes,
        dtype=np.uint8
    ).astype(np.int64)

    encoded = encoded[:MAX_HTML_LENGTH] + 1

    padded = np.zeros(
        MAX_HTML_LENGTH,
        dtype=np.int64
    )

    length = min(len(encoded), MAX_HTML_LENGTH)
    padded[:length] = encoded[:length]

    return padded


class RawHTMLDataset(Dataset):
    def __init__(self, dataframe):
        self.archive_paths = dataframe["archive_path"].tolist()
        self.labels = dataframe["label_id"].astype("float32").to_numpy()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        html_bytes = read_html_from_archive(
            self.archive_paths[index]
        )

        html_tensor = torch.tensor(
            encode_html(html_bytes),
            dtype=torch.long
        )

        label_tensor = torch.tensor(
            self.labels[index],
            dtype=torch.float32
        )

        return html_tensor, label_tensor

In [69]:
# CELLULE 61 - Séparation des ensembles HTML

raw_html_train_df = raw_html_df[
    raw_html_df["split"] == "train"
].copy()

raw_html_validation_df = raw_html_df[
    raw_html_df["split"] == "validation"
].copy()

raw_html_test_df = raw_html_df[
    raw_html_df["split"] == "test"
].copy()

html_train_dataset = RawHTMLDataset(raw_html_train_df)
html_validation_dataset = RawHTMLDataset(raw_html_validation_df)
html_test_dataset = RawHTMLDataset(raw_html_test_df)

html_train_loader = DataLoader(
    html_train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

html_validation_loader = DataLoader(
    html_validation_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

html_test_loader = DataLoader(
    html_test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("Train :", len(html_train_dataset))
print("Validation :", len(html_validation_dataset))
print("Test :", len(html_test_dataset))

Train : 42210
Validation : 8194
Test : 17098


In [70]:
# CELLULE 62 - CNN 1D sur HTML brut

class HTMLCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=257,
            embedding_dim=64,
            padding_idx=0
        )

        self.convolution = nn.Sequential(
            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=7,
                padding=3
            ),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(
                in_channels=128,
                out_channels=128,
                kernel_size=7,
                padding=3
            ),
            nn.ReLU(),

            nn.AdaptiveMaxPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, 1)
        )

    def forward(self, inputs):
        embedded = self.embedding(inputs)
        embedded = embedded.transpose(1, 2)

        features = self.convolution(embedded)

        return self.classifier(features).squeeze(1)


html_cnn_model = HTMLCNN().to(device)

html_positive_count = raw_html_train_df["label_id"].sum()
html_negative_count = (
    len(raw_html_train_df) - html_positive_count
)

html_pos_weight = torch.tensor(
    [html_negative_count / html_positive_count],
    dtype=torch.float32,
    device=device
)

html_criterion = nn.BCEWithLogitsLoss(
    pos_weight=html_pos_weight
)

html_optimizer = torch.optim.AdamW(
    html_cnn_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print("Modèle HTML créé.")
print(
    "Paramètres :",
    sum(
        parameter.numel()
        for parameter in html_cnn_model.parameters()
    )
)

Modèle HTML créé.
Paramètres : 189121


In [71]:
# CELLULE 63 - Prédictions du CNN HTML

def predict_html_cnn(model, loader):
    model.eval()

    probabilities = []
    labels = []

    with torch.no_grad():
        for inputs, batch_labels in loader:
            inputs = inputs.to(device)

            logits = model(inputs)
            batch_probabilities = torch.sigmoid(logits)

            probabilities.extend(
                batch_probabilities.cpu().numpy()
            )
            labels.extend(
                batch_labels.numpy()
            )

    return (
        np.asarray(labels),
        np.asarray(probabilities)
    )

In [ ]:
# CELLULE 64 - Entraînement du CNN HTML

HTML_EPOCHS = 10
HTML_PATIENCE = 3

best_html_validation_pr_auc = -np.inf
best_html_state = None
html_epochs_without_improvement = 0

for epoch in range(HTML_EPOCHS):
    html_cnn_model.train()
    epoch_loss = 0.0

    for inputs, labels in html_train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        html_optimizer.zero_grad()

        logits = html_cnn_model(inputs)
        loss = html_criterion(logits, labels)

        loss.backward()
        html_optimizer.step()

        epoch_loss += loss.item() * len(labels)

    epoch_loss /= len(html_train_loader.dataset)

    validation_labels, validation_probabilities = (
        predict_html_cnn(
            html_cnn_model,
            html_validation_loader
        )
    )

    validation_pr_auc_html = average_precision_score(
        validation_labels,
        validation_probabilities
    )

    print(
        f"Epoch {epoch + 1}/{HTML_EPOCHS} | "
        f"loss={epoch_loss:.5f} | "
        f"validation PR-AUC={validation_pr_auc_html:.6f}"
    )

    if validation_pr_auc_html > best_html_validation_pr_auc:
        best_html_validation_pr_auc = validation_pr_auc_html
        best_html_state = {
            key: value.detach().cpu().clone()
            for key, value in html_cnn_model.state_dict().items()
        }
        html_epochs_without_improvement = 0
    else:
        html_epochs_without_improvement += 1

    if html_epochs_without_improvement >= HTML_PATIENCE:
        print("Early stopping.")
        break

if best_html_state is None:
    raise RuntimeError(
        "Aucun état HTML valide n'a été sauvegardé."
    )

html_cnn_model.load_state_dict(best_html_state)
html_cnn_model.to(device)

print(
    "Meilleure validation PR-AUC HTML :",
    round(best_html_validation_pr_auc, 6)
)

Epoch 1/10 | loss=0.35393 | validation PR-AUC=0.949354


In [72]:
# CELLULE - Contrôle train, validation et test du CNN HTML

train_html_labels, train_html_probabilities = predict_html_cnn(
    html_cnn_model,
    html_train_loader
)

validation_html_labels, validation_html_probabilities = predict_html_cnn(
    html_cnn_model,
    html_validation_loader
)

test_html_labels, test_html_probabilities = predict_html_cnn(
    html_cnn_model,
    html_test_loader
)

print(
    "Train PR-AUC :",
    round(
        average_precision_score(
            train_html_labels,
            train_html_probabilities
        ),
        6
    )
)

print(
    "Validation PR-AUC :",
    round(
        average_precision_score(
            validation_html_labels,
            validation_html_probabilities
        ),
        6
    )
)

print(
    "Test PR-AUC :",
    round(
        average_precision_score(
            test_html_labels,
            test_html_probabilities
        ),
        6
    )
)

print(
    "Test ROC-AUC :",
    round(
        roc_auc_score(
            test_html_labels,
            test_html_probabilities
        ),
        6
    )
)

Train PR-AUC : 0.999306
Validation PR-AUC : 0.982165
Test PR-AUC : 0.979966
Test ROC-AUC : 0.980595


In [73]:
# CELLULE - Seuil HTML sélectionné sur validation

html_threshold_rows = []

for threshold in np.arange(0.10, 0.91, 0.05):
    validation_predictions = (
        validation_html_probabilities >= threshold
    ).astype("int8")

    html_threshold_rows.append({
        "threshold": threshold,
        "f1_phishing": f1_score(
            validation_html_labels,
            validation_predictions,
            zero_division=0
        )
    })

html_threshold_df = pd.DataFrame(html_threshold_rows)

display(html_threshold_df)

selected_html_threshold = html_threshold_df.loc[
    html_threshold_df["f1_phishing"].idxmax(),
    "threshold"
]

print("Seuil HTML sélectionné :", selected_html_threshold)

,threshold,f1_phishing
0,0.10,0.924109
1,0.15,0.928389
2,0.20,0.930316
3,0.25,0.931488
4,0.30,0.932594
5,0.35,0.933137
6,0.40,0.932070
7,0.45,0.931883
8,0.50,0.931818
9,0.55,0.929627


Seuil HTML sélectionné : 0.3500000000000001


In [74]:
# CELLULE - Évaluation finale HTML

html_test_predictions = (
    test_html_probabilities >= selected_html_threshold
).astype("int8")

tn, fp, fn, tp = confusion_matrix(
    test_html_labels,
    html_test_predictions,
    labels=[0, 1]
).ravel()

html_cnn_final_results = pd.DataFrame([{
    "model": "cnn_raw_html",
    "threshold": selected_html_threshold,
    "test_pr_auc": average_precision_score(
        test_html_labels,
        test_html_probabilities
    ),
    "test_roc_auc": roc_auc_score(
        test_html_labels,
        test_html_probabilities
    ),
    "test_precision_phishing": precision_score(
        test_html_labels,
        html_test_predictions,
        zero_division=0
    ),
    "test_recall_phishing": recall_score(
        test_html_labels,
        html_test_predictions,
        zero_division=0
    ),
    "test_f1_phishing": f1_score(
        test_html_labels,
        html_test_predictions,
        zero_division=0
    ),
    "false_positives": fp,
    "false_negatives": fn,
    "true_negatives": tn,
    "true_positives": tp
}])

display(html_cnn_final_results)

,model,threshold,test_pr_auc,test_roc_auc,test_precision_phishing,test_recall_phishing,test_f1_phishing,false_positives,false_negatives,true_negatives,true_positives
0,cnn_raw_html,0.35,0.979966,0.980595,0.960801,0.909066,0.934218,270,662,9548,6618


In [75]:
# CELLULE 66 - Dataset screenshots bruts

from PIL import Image

SCREENSHOT_SIZE = 128

raw_screenshot_df = features_df[
    ["archive_filename", "label_id", "split"]
].copy()

raw_screenshot_df["archive_path"] = raw_screenshot_df[
    "archive_filename"
].apply(lambda name: str(DATASET_ROOT / name))

def read_screenshot_from_archive(archive_path):
    try:
        with zipfile.ZipFile(archive_path, "r") as archive:
            image_bytes = archive.read("artifacts/screenshot.png")

        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        image = image.resize(
            (SCREENSHOT_SIZE, SCREENSHOT_SIZE),
            Image.Resampling.BILINEAR
        )

        image_array = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        image_array = np.transpose(
            image_array,
            (2, 0, 1)
        )

        return image_array

    except (
        KeyError,
        OSError,
        EOFError,
        zipfile.BadZipFile
    ):
        return np.zeros(
            (3, SCREENSHOT_SIZE, SCREENSHOT_SIZE),
            dtype=np.float32
        )


class RawScreenshotDataset(Dataset):
    def __init__(self, dataframe):
        self.archive_paths = dataframe["archive_path"].tolist()
        self.labels = dataframe["label_id"].astype(
            "float32"
        ).to_numpy()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image_array = read_screenshot_from_archive(
            self.archive_paths[index]
        )

        image_tensor = torch.tensor(
            image_array,
            dtype=torch.float32
        )

        label_tensor = torch.tensor(
            self.labels[index],
            dtype=torch.float32
        )

        return image_tensor, label_tensor

In [76]:
# CELLULE 67 - Train, validation et test screenshots

raw_screenshot_train_df = raw_screenshot_df[
    raw_screenshot_df["split"] == "train"
].copy()

raw_screenshot_validation_df = raw_screenshot_df[
    raw_screenshot_df["split"] == "validation"
].copy()

raw_screenshot_test_df = raw_screenshot_df[
    raw_screenshot_df["split"] == "test"
].copy()

screenshot_train_dataset = RawScreenshotDataset(
    raw_screenshot_train_df
)

screenshot_validation_dataset = RawScreenshotDataset(
    raw_screenshot_validation_df
)

screenshot_test_dataset = RawScreenshotDataset(
    raw_screenshot_test_df
)

screenshot_train_loader = DataLoader(
    screenshot_train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

screenshot_validation_loader = DataLoader(
    screenshot_validation_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0
)

screenshot_test_loader = DataLoader(
    screenshot_test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0
)

print("Train :", len(screenshot_train_dataset))
print("Validation :", len(screenshot_validation_dataset))
print("Test :", len(screenshot_test_dataset))

Train : 42210
Validation : 8194
Test : 17098


In [77]:
# CELLULE 68 - CNN sur les pixels des screenshots

class ScreenshotCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, 1)
        )

    def forward(self, inputs):
        features = self.features(inputs)
        return self.classifier(features).squeeze(1)


screenshot_cnn_model = ScreenshotCNN().to(device)

screenshot_positive_count = (
    raw_screenshot_train_df["label_id"].sum()
)

screenshot_negative_count = (
    len(raw_screenshot_train_df)
    - screenshot_positive_count
)

screenshot_pos_weight = torch.tensor(
    [screenshot_negative_count / screenshot_positive_count],
    dtype=torch.float32,
    device=device
)

screenshot_criterion = nn.BCEWithLogitsLoss(
    pos_weight=screenshot_pos_weight
)

screenshot_optimizer = torch.optim.AdamW(
    screenshot_cnn_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print("CNN screenshot créé.")
print(
    "Paramètres :",
    sum(
        parameter.numel()
        for parameter in screenshot_cnn_model.parameters()
    )
)

CNN screenshot créé.
Paramètres : 93825


In [78]:
# CELLULE 69 - Fonctions et entraînement du CNN screenshot

def predict_screenshot_cnn(model, loader):
    model.eval()

    probabilities = []
    labels = []

    with torch.no_grad():
        for inputs, batch_labels in loader:
            inputs = inputs.to(device)

            logits = model(inputs)
            batch_probabilities = torch.sigmoid(logits)

            probabilities.extend(
                batch_probabilities.cpu().numpy()
            )
            labels.extend(
                batch_labels.numpy()
            )

    return (
        np.asarray(labels),
        np.asarray(probabilities)
    )


SCREENSHOT_EPOCHS = 10
SCREENSHOT_PATIENCE = 3

best_screenshot_validation_pr_auc = -np.inf
best_screenshot_state = None
screenshot_epochs_without_improvement = 0

for epoch in range(SCREENSHOT_EPOCHS):
    screenshot_cnn_model.train()
    epoch_loss = 0.0

    for inputs, labels in screenshot_train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        screenshot_optimizer.zero_grad()

        logits = screenshot_cnn_model(inputs)
        loss = screenshot_criterion(logits, labels)

        loss.backward()
        screenshot_optimizer.step()

        epoch_loss += loss.item() * len(labels)

    epoch_loss /= len(screenshot_train_loader.dataset)

    validation_labels, validation_probabilities = (
        predict_screenshot_cnn(
            screenshot_cnn_model,
            screenshot_validation_loader
        )
    )

    validation_pr_auc_screenshot = average_precision_score(
        validation_labels,
        validation_probabilities
    )

    print(
        f"Epoch {epoch + 1}/{SCREENSHOT_EPOCHS} | "
        f"loss={epoch_loss:.5f} | "
        f"validation PR-AUC="
        f"{validation_pr_auc_screenshot:.6f}"
    )

    if (
        validation_pr_auc_screenshot
        > best_screenshot_validation_pr_auc
    ):
        best_screenshot_validation_pr_auc = (
            validation_pr_auc_screenshot
        )

        best_screenshot_state = {
            key: value.detach().cpu().clone()
            for key, value in (
                screenshot_cnn_model.state_dict().items()
            )
        }

        screenshot_epochs_without_improvement = 0
    else:
        screenshot_epochs_without_improvement += 1

    if (
        screenshot_epochs_without_improvement
        >= SCREENSHOT_PATIENCE
    ):
        print("Early stopping.")
        break

if best_screenshot_state is None:
    raise RuntimeError(
        "Aucun état valide du CNN screenshot n'a été sauvegardé."
    )

screenshot_cnn_model.load_state_dict(
    best_screenshot_state
)

screenshot_cnn_model.to(device)

print(
    "Meilleure validation PR-AUC screenshot :",
    round(
        best_screenshot_validation_pr_auc,
        6
    )
)

Epoch 1/10 | loss=0.62746 | validation PR-AUC=0.632880
Epoch 2/10 | loss=0.58831 | validation PR-AUC=0.663198
Epoch 3/10 | loss=0.55835 | validation PR-AUC=0.674753
Epoch 4/10 | loss=0.52379 | validation PR-AUC=0.683913
Epoch 5/10 | loss=0.49307 | validation PR-AUC=0.818255
Epoch 6/10 | loss=0.46404 | validation PR-AUC=0.755334
Epoch 7/10 | loss=0.45257 | validation PR-AUC=0.586598
Epoch 8/10 | loss=0.43274 | validation PR-AUC=0.828711
Epoch 9/10 | loss=0.41398 | validation PR-AUC=0.806926
Epoch 10/10 | loss=0.40169 | validation PR-AUC=0.854149
Meilleure validation PR-AUC screenshot : 0.854149


In [79]:
# CELLULE 70 - Seuil sélectionné sur validation

screenshot_validation_labels, screenshot_validation_probabilities = (
    predict_screenshot_cnn(
        screenshot_cnn_model,
        screenshot_validation_loader
    )
)

screenshot_test_labels, screenshot_test_probabilities = (
    predict_screenshot_cnn(
        screenshot_cnn_model,
        screenshot_test_loader
    )
)

screenshot_threshold_rows = []

for threshold in np.arange(0.10, 0.91, 0.05):
    validation_predictions = (
        screenshot_validation_probabilities >= threshold
    ).astype("int8")

    screenshot_threshold_rows.append({
        "threshold": threshold,
        "f1_phishing": f1_score(
            screenshot_validation_labels,
            validation_predictions,
            zero_division=0
        )
    })

screenshot_threshold_df = pd.DataFrame(
    screenshot_threshold_rows
)

display(screenshot_threshold_df)

selected_screenshot_threshold = screenshot_threshold_df.loc[
    screenshot_threshold_df["f1_phishing"].idxmax(),
    "threshold"
]

print(
    "Seuil screenshot sélectionné :",
    selected_screenshot_threshold
)

,threshold,f1_phishing
0,0.10,0.761270
1,0.15,0.770185
2,0.20,0.744762
3,0.25,0.754448
4,0.30,0.757026
5,0.35,0.762364
6,0.40,0.759565
7,0.45,0.753776
8,0.50,0.745410
9,0.55,0.686329


Seuil screenshot sélectionné : 0.15000000000000002


In [80]:
# CELLULE 71 - Évaluation finale du CNN screenshot

screenshot_test_predictions = (
    screenshot_test_probabilities
    >= selected_screenshot_threshold
).astype("int8")

tn, fp, fn, tp = confusion_matrix(
    screenshot_test_labels,
    screenshot_test_predictions,
    labels=[0, 1]
).ravel()

screenshot_cnn_results = pd.DataFrame([{
    "model": "cnn_raw_screenshot",
    "threshold": selected_screenshot_threshold,
    "validation_pr_auc": average_precision_score(
        screenshot_validation_labels,
        screenshot_validation_probabilities
    ),
    "test_pr_auc": average_precision_score(
        screenshot_test_labels,
        screenshot_test_probabilities
    ),
    "test_roc_auc": roc_auc_score(
        screenshot_test_labels,
        screenshot_test_probabilities
    ),
    "test_precision_phishing": precision_score(
        screenshot_test_labels,
        screenshot_test_predictions,
        zero_division=0
    ),
    "test_recall_phishing": recall_score(
        screenshot_test_labels,
        screenshot_test_predictions,
        zero_division=0
    ),
    "test_f1_phishing": f1_score(
        screenshot_test_labels,
        screenshot_test_predictions,
        zero_division=0
    ),
    "false_positives": fp,
    "false_negatives": fn,
    "true_negatives": tn,
    "true_positives": tp
}])

display(screenshot_cnn_results)

,model,threshold,validation_pr_auc,test_pr_auc,test_roc_auc,test_precision_phishing,test_recall_phishing,test_f1_phishing,false_positives,false_negatives,true_negatives,true_positives
0,cnn_raw_screenshot,0.15,0.854149,0.834389,0.821571,0.523776,0.918407,0.667099,6079,594,3739,6686


In [81]:
# CELLULE 72 - Vérification et fusion tardive des modèles

from itertools import combinations

# Vérifier que les exemples sont dans le même ordre
assert np.array_equal(url_validation_labels, technical_validation_labels)
assert np.array_equal(url_validation_labels, validation_html_labels)
assert np.array_equal(url_validation_labels, screenshot_validation_labels)

assert np.array_equal(url_test_labels, technical_test_labels)
assert np.array_equal(url_test_labels, test_html_labels)
assert np.array_equal(url_test_labels, screenshot_test_labels)

fusion_validation_labels = url_validation_labels.astype("int8")
fusion_test_labels = url_test_labels.astype("int8")

validation_probabilities_by_modality = {
    "url": url_validation_probabilities,
    "technical": technical_validation_probabilities,
    "html": validation_html_probabilities,
    "screenshot": screenshot_validation_probabilities
}

test_probabilities_by_modality = {
    "url": url_test_probabilities,
    "technical": technical_test_probabilities,
    "html": test_html_probabilities,
    "screenshot": screenshot_test_probabilities
}

print("Alignement des modalités vérifié.")

Alignement des modalités vérifié.


In [82]:
# CELLULE 73 - Évaluation d'une combinaison de modalités

def evaluate_late_fusion(modalities):
    validation_probabilities = np.mean(
        [
            validation_probabilities_by_modality[modality]
            for modality in modalities
        ],
        axis=0
    )

    test_probabilities = np.mean(
        [
            test_probabilities_by_modality[modality]
            for modality in modalities
        ],
        axis=0
    )

    threshold_rows = []

    for threshold in np.arange(0.10, 0.91, 0.05):
        validation_predictions = (
            validation_probabilities >= threshold
        ).astype("int8")

        threshold_rows.append({
            "threshold": threshold,
            "f1_phishing": f1_score(
                fusion_validation_labels,
                validation_predictions,
                zero_division=0
            )
        })

    threshold_df = pd.DataFrame(threshold_rows)

    selected_threshold = threshold_df.loc[
        threshold_df["f1_phishing"].idxmax(),
        "threshold"
    ]

    test_predictions = (
        test_probabilities >= selected_threshold
    ).astype("int8")

    tn, fp, fn, tp = confusion_matrix(
        fusion_test_labels,
        test_predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "modalities": " + ".join(modalities),
        "modality_count": len(modalities),
        "threshold": selected_threshold,
        "validation_pr_auc": average_precision_score(
            fusion_validation_labels,
            validation_probabilities
        ),
        "test_pr_auc": average_precision_score(
            fusion_test_labels,
            test_probabilities
        ),
        "test_roc_auc": roc_auc_score(
            fusion_test_labels,
            test_probabilities
        ),
        "test_precision_phishing": precision_score(
            fusion_test_labels,
            test_predictions,
            zero_division=0
        ),
        "test_recall_phishing": recall_score(
            fusion_test_labels,
            test_predictions,
            zero_division=0
        ),
        "test_f1_phishing": f1_score(
            fusion_test_labels,
            test_predictions,
            zero_division=0
        ),
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "true_positives": tp
    }

In [83]:
# CELLULE 74 - Combinaisons à comparer

fusion_combinations = [
    ("url",),
    ("technical",),
    ("html",),
    ("screenshot",),

    ("url", "technical"),
    ("url", "html"),
    ("url", "screenshot"),
    ("technical", "html"),
    ("technical", "screenshot"),
    ("html", "screenshot"),

    ("url", "html", "technical"),
    ("url", "html", "screenshot"),
    ("url", "technical", "screenshot"),

    ("url", "technical", "html", "screenshot")
]

late_fusion_results = []

for modalities in fusion_combinations:
    print("Évaluation :", " + ".join(modalities))

    late_fusion_results.append(
        evaluate_late_fusion(modalities)
    )

late_fusion_results_df = pd.DataFrame(
    late_fusion_results
)

display(
    late_fusion_results_df.sort_values(
        "test_pr_auc",
        ascending=False
    ).reset_index(drop=True)
)

RESULTS_DIR = Path.cwd() / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
late_fusion_results_df.sort_values(
    'test_pr_auc',
    ascending=False
).reset_index(drop=True).to_csv(
    RESULTS_DIR / 'late_fusion_results.csv',
    index=False
)
print('Résultats sauvegardés :', RESULTS_DIR / 'late_fusion_results.csv')

Évaluation : url
Évaluation : technical
Évaluation : html
Évaluation : screenshot
Évaluation : url + technical
Évaluation : url + html
Évaluation : url + screenshot
Évaluation : technical + html
Évaluation : technical + screenshot
Évaluation : html + screenshot
Évaluation : url + html + technical
Évaluation : url + html + screenshot
Évaluation : url + technical + screenshot
Évaluation : url + technical + html + screenshot


,modalities,modality_count,threshold,validation_pr_auc,test_pr_auc,test_roc_auc,test_precision_phishing,test_recall_phishing,test_f1_phishing,false_positives,false_negatives,true_negatives,true_positives
0,url + html + technical,3,0.40,0.994636,0.996805,0.996675,0.989056,0.968269,0.978552,78,231,9740,7049
1,url + html,2,0.50,0.989363,0.996522,0.996967,0.995476,0.937088,0.965400,31,458,9787,6822
2,url + technical + html + screenshot,4,0.40,0.994257,0.995088,0.994574,0.990077,0.959341,0.974466,70,296,9748,6984
3,url + html + screenshot,3,0.45,0.988600,0.992814,0.991820,0.992851,0.934753,0.962926,49,475,9769,6805
4,url + technical,2,0.50,0.983620,0.992306,0.991841,0.989858,0.938462,0.963475,70,448,9748,6832
5,url + technical + screenshot,3,0.45,0.983982,0.990965,0.989794,0.985626,0.941896,0.963265,100,423,9718,6857
6,url,1,0.40,0.966195,0.987511,0.988528,0.973704,0.910440,0.941009,179,652,9639,6628
7,technical + html,2,0.50,0.988524,0.985978,0.983771,0.986012,0.910165,0.946571,94,654,9724,6626
8,url + screenshot,2,0.40,0.962411,0.982838,0.980972,0.971698,0.933791,0.952368,198,482,9620,6798
9,html,1,0.35,0.982165,0.979966,0.980595,0.960801,0.909066,0.934218,270,662,9548,6618
